In [ ]:
#Just to check Python version

import sys

print("Welcome to ProductPilot AI!")
print("Python version:", sys.version)

Welcome to ProductPilot AI!
Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [18]:
# Installing Dependencies

%pip install -q openai-agents

In [ ]:
#SDK Import test

from agents import Agent, Runner, function_tool

print("OpenAI Agents SDK installed successfully!")

OpenAI Agents SDK installed successfully!


In [20]:
# Gemini Configuration check

from google.colab import userdata

from agents import (
    Agent,
    Runner,
    AsyncOpenAI,
    OpenAIChatCompletionsModel,
    set_tracing_disabled
)

# Load API key securely
gemini_key = userdata.get("GEMINI_API_KEY")

# Connect to Gemini
client = AsyncOpenAI(
    api_key=gemini_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# Configure our working Gemini model
model = OpenAIChatCompletionsModel(
    model="gemini-2.5-flash-lite",
    openai_client=client
)

# Disable OpenAI tracing
set_tracing_disabled(True)

print("Gemini configured successfully!")

Gemini configured successfully!


In [ ]:
# Create our first specialized agent
feedback_agent = Agent(
    name="Feedback Analysis Agent",

    instructions="""
    You are the Feedback Analysis Agent for ProductPilot AI.

    Analyze customer feedback for StudyFlow,
    a student productivity application.

    Identify:
    1. Feedback Category
    2. Main User Problem
    3. Requested Feature
    4. Recommended Action

    Keep responses concise and structured.
    """,

    model=model
)

# Run the agent
result = await Runner.run(
    feedback_agent,
    "The StudyFlow app doesn't remind me about "
    "upcoming assignment deadlines."
)

print("=== PRODUCTPILOT AI ===")
print("Agent:", feedback_agent.name)
print()
print(result.final_output)

=== PRODUCTPILOT AI ===
Agent: Feedback Analysis Agent

**1. Feedback Category:** Missing Feature

**2. Main User Problem:** Forgetting assignment deadlines due to lack of timely reminders.

**3. Requested Feature:** Deadline notification system.

**4. Recommended Action:** Implement a feature that allows users to set and receive reminders for assignment deadlines.


In [ ]:
import pandas as pd

# Fictional customer feedback for StudyFlow
feedback_data = [
    [1, "Please add reminders for upcoming assignment deadlines."],
    [2, "I often forget deadlines because the app doesn't notify me."],
    [3, "I want a calendar showing all my exams and assignments."],
    [4, "The app becomes slow when I add too many tasks."],
    [5, "Deadline notifications would help me submit assignments on time."],
    [6, "Please add a dark mode for studying at night."],
    [7, "I need a calendar to organize my weekly study schedule."],
    [8, "The app crashes sometimes when I open my task list."],
    [9, "It would be useful to receive reminders one day before deadlines."],
    [10, "Please allow me to track how many hours I study daily."],
    [11, "I want to see all upcoming deadlines in a calendar."],
    [12, "The app takes too long to load when I have many assignments."],
    [13, "Please add customizable reminders for important tasks."],
    [14, "A study progress dashboard would help me stay organized."],
    [15, "I want to sync my tasks across my phone and laptop."]
]

# Create a DataFrame
df = pd.DataFrame(
    feedback_data,
    columns=["feedback_id", "feedback_text"]
)

# Save dataset as CSV
df.to_csv("customer_feedback.csv", index=False)

print("Customer feedback dataset created successfully!")
print("Total feedback entries:", len(df))

display(df)

Customer feedback dataset created successfully!
Total feedback entries: 15


,feedback_id,feedback_text
0,1,Please add reminders for upcoming assignment d...
1,2,I often forget deadlines because the app doesn...
2,3,I want a calendar showing all my exams and ass...
3,4,The app becomes slow when I add too many tasks.
4,5,Deadline notifications would help me submit as...
5,6,Please add a dark mode for studying at night.
6,7,I need a calendar to organize my weekly study ...
7,8,The app crashes sometimes when I open my task ...
8,9,It would be useful to receive reminders one da...
9,10,Please allow me to track how many hours I stud...


In [84]:
from agents import function_tool
import pandas as pd

# Create our first real tool
@function_tool
def read_customer_feedback() -> str:
    """
    Read customer feedback from the StudyFlow CSV dataset.

    Returns:
        All customer feedback records as text.
    """

    try:
        # Read the CSV file
        df = pd.read_csv("customer_feedback.csv")

        # Convert feedback into readable text
        feedback_text = df.to_string(index=False)

        return feedback_text

    except FileNotFoundError:
        return "Error: Customer feedback dataset not found."

    except Exception as e:
        return f"Error reading feedback: {str(e)}"


# Update our existing agent with the tool
feedback_agent = Agent(
    name="Feedback Analysis Agent",

    instructions="""
    You are the Feedback Analysis Agent for ProductPilot AI.

    Your responsibility is to analyze customer feedback
    for StudyFlow, a student productivity application.

    IMPORTANT:
    Always use the read_customer_feedback tool
    before analyzing customer feedback.

    After reading the dataset:

    1. Identify recurring customer complaints.
    2. Identify requested features.
    3. Group similar feedback.
    4. Count how frequently each issue appears.
    5. Recommend the top 3 product improvements.

    Base your analysis only on the actual dataset.

    Do not invent customer feedback or statistics.

    Present your findings clearly.
    """,

    model=model,

    # Connect our Python tool to the agent
    tools=[read_customer_feedback]
)

print("Feedback Analysis Agent updated with CSV tool!")

Feedback Analysis Agent updated with CSV tool!


In [85]:
# Run the agent on our complete feedback dataset
result = await Runner.run(
    feedback_agent,
    "Analyze all customer feedback and identify "
    "the most frequently requested features."
)

print("=== PRODUCTPILOT AI ===")
print("FEEDBACK ANALYSIS REPORT\n")

print(result.final_output)

# Verify tool execution
print("\n=== TOOL EXECUTION LOG ===")

for item in result.new_items:
    print(type(item).__name__)

=== PRODUCTPILOT AI ===
FEEDBACK ANALYSIS REPORT

Here's an analysis of the customer feedback for StudyFlow:

**Recurring Complaints:**

*   **Performance Issues:** Several users reported that the app becomes slow or crashes when dealing with a large number of tasks or assignments. (3 instances)
*   **Lack of Deadline Reminders:** A significant number of users are missing deadlines due to the absence of timely notifications. (5 instances)

**Requested Features:**

*   **Deadline Reminders:** Users want customizable reminders for upcoming assignment and task deadlines, including options for advance notifications (e.g., one day before). (5 instances)
*   **Calendar Integration:** Users want a calendar view to organize exams, assignments, and their weekly study schedule, displaying all upcoming deadlines. (3 instances)
*   **Study Hour Tracking:** A feature to track daily study hours is requested. (1 instance)
*   **Dark Mode:** Users have requested a dark mode option. (1 instance)
*   **

In [86]:
import pandas as pd

# Primary category for each feedback record
feedback_categories = {
    1: "Deadline Reminders",
    2: "Deadline Reminders",
    3: "Calendar",
    4: "Performance & Stability",
    5: "Deadline Reminders",
    6: "Dark Mode",
    7: "Calendar",
    8: "Performance & Stability",
    9: "Deadline Reminders",
    10: "Study Hour Tracking",
    11: "Calendar",
    12: "Performance & Stability",
    13: "Deadline Reminders",
    14: "Study Progress Dashboard",
    15: "Cross-Device Sync"
}

# Read our existing dataset
df = pd.read_csv("customer_feedback.csv")

# Assign categories using feedback IDs
df["category"] = df["feedback_id"].map(feedback_categories)

# Check for missing classifications
if df["category"].isna().any():
    raise ValueError("Some feedback records have no category!")

# Save the updated dataset
df.to_csv("customer_feedback.csv", index=False)

# Calculate exact category frequencies
category_counts = (
    df["category"]
    .value_counts()
    .rename_axis("Feature / Issue")
    .reset_index(name="Frequency")
)

print("=== VERIFIED FEEDBACK ANALYSIS ===")
display(category_counts)

print("\nTotal feedback records:", len(df))
print("Total categorized records:", category_counts["Frequency"].sum())

=== VERIFIED FEEDBACK ANALYSIS ===


,Feature / Issue,Frequency
0,Deadline Reminders,5
1,Calendar,3
2,Performance & Stability,3
3,Dark Mode,1
4,Study Hour Tracking,1
5,Study Progress Dashboard,1
6,Cross-Device Sync,1



Total feedback records: 15
Total categorized records: 15


In [87]:
from agents import function_tool
import pandas as pd

@function_tool
def get_feedback_statistics() -> str:
    """
    Calculate verified feedback category frequencies
    from the StudyFlow customer feedback dataset.

    Returns:
        Category counts and total feedback records.
    """

    try:
        df = pd.read_csv("customer_feedback.csv")

        # Check required columns
        required_columns = {"feedback_id", "category"}

        if not required_columns.issubset(df.columns):
            return "Error: Required dataset columns are missing."

        # Check for missing categories
        if df["category"].isna().any():
            return "Error: Some feedback records are not categorized."

        # Calculate exact frequencies
        counts = df["category"].value_counts()

        # Prepare readable results
        report = "VERIFIED CUSTOMER FEEDBACK STATISTICS\n\n"

        for category, count in counts.items():
            report += f"{category}: {count}\n"

        report += f"\nTotal Feedback Records: {len(df)}"

        return report

    except FileNotFoundError:
        return "Error: Customer feedback CSV file not found."

    except Exception as e:
        return f"Error calculating statistics: {str(e)}"


print("Tool 2 created successfully!")

Tool 2 created successfully!


In [88]:
# Update Feedback Analysis Agent with both tools
feedback_agent = Agent(
    name="Feedback Analysis Agent",

    instructions="""
    You are the Feedback Analysis Agent for ProductPilot AI.

    Analyze customer feedback for StudyFlow.

    You MUST use both available tools:

    1. read_customer_feedback:
       Read and understand customer feedback.

    2. get_feedback_statistics:
       Retrieve verified feedback category frequencies.

    Your responsibilities:

    - Identify major customer problems.
    - Summarize requested features.
    - Report exact category frequencies.
    - Recommend the top 3 product improvements.

    IMPORTANT:
    Use the statistics tool for all numerical counts.
    Never invent feedback frequencies.
    Do not double-count categories.
    """,

    model=model,

    tools=[
        read_customer_feedback,
        get_feedback_statistics
    ]
)

# Run the updated agent
result = await Runner.run(
    feedback_agent,
    "Analyze StudyFlow customer feedback, "
    "report verified category frequencies, "
    "and recommend the top 3 improvements."
)

print("=== PRODUCTPILOT AI ===")
print("FEEDBACK ANALYSIS REPORT\n")

print(result.final_output)

print("\n=== TOOL EXECUTION LOG ===")

for item in result.new_items:
    if item.type == "tool_call_item":
        print("Tool called:", item.raw_item.name)

    elif item.type == "tool_call_output_item":
        print("Tool output received")

    else:
        print(type(item).__name__)


=== PRODUCTPILOT AI ===
FEEDBACK ANALYSIS REPORT

The major customer problems are:

*   **Deadline Reminders:** Many users are forgetting deadlines due to a lack of timely notifications and reminders.
*   **Performance & Stability:** The app experiences slowness and crashes, particularly when users have a large number of tasks or assignments.
*   **Calendar Integration:** Users desire a calendar view to organize exams, assignments, and their weekly study schedule.

The main requested feature is a more robust notification system for assignment deadlines, with several users specifically asking for customizable and timely reminders.

Verified category frequencies are:

*   Deadline Reminders: 5
*   Calendar: 3
*   Performance & Stability: 3
*   Dark Mode: 1
*   Study Hour Tracking: 1
*   Study Progress Dashboard: 1
*   Cross-Device Sync: 1

Top 3 product improvements:

1.  **Implement a reliable and customizable deadline reminder system:** This is the most frequently requested feature and

In [ ]:
import pandas as pd

# Fictional competitor information
competitor_data = [
    {
        "product": "StudyFlow",
        "deadline_reminders": 0,
        "calendar": 0,
        "dark_mode": 0,
        "study_tracking": 1,
        "cross_device_sync": 0,
        "performance_rating": 3
    },
    {
        "product": "TaskMaster",
        "deadline_reminders": 1,
        "calendar": 1,
        "dark_mode": 1,
        "study_tracking": 0,
        "cross_device_sync": 1,
        "performance_rating": 4
    },
    {
        "product": "EduPlanner",
        "deadline_reminders": 1,
        "calendar": 1,
        "dark_mode": 0,
        "study_tracking": 1,
        "cross_device_sync": 1,
        "performance_rating": 5
    },
    {
        "product": "FocusMate",
        "deadline_reminders": 0,
        "calendar": 1,
        "dark_mode": 1,
        "study_tracking": 1,
        "cross_device_sync": 0,
        "performance_rating": 4
    }
]

# Create DataFrame
competitor_df = pd.DataFrame(competitor_data)

# Save dataset
competitor_df.to_csv("competitor_data.csv", index=False)

print("Competitor dataset created successfully!")
print("Total products:", len(competitor_df))

display(competitor_df)

Competitor dataset created successfully!
Total products: 4


,product,deadline_reminders,calendar,dark_mode,study_tracking,cross_device_sync,performance_rating
0,StudyFlow,0,0,0,1,0,3
1,TaskMaster,1,1,1,0,1,4
2,EduPlanner,1,1,0,1,1,5
3,FocusMate,0,1,1,1,0,4


In [75]:
from agents import Agent, function_tool
import pandas as pd

# TOOL 3: Retrieve competitor information
@function_tool
def get_competitor_data() -> str:
    """
    Read competitor product features and performance ratings
    from the StudyFlow competitor dataset.

    Returns:
        Competitor comparison data as text.
    """

    try:
        df = pd.read_csv("competitor_data.csv")

        if df.empty:
            return "Error: Competitor dataset is empty."

        return df.to_string(index=False)

    except FileNotFoundError:
        return "Error: Competitor dataset not found."

    except Exception as e:
        return f"Error reading competitor data: {str(e)}"


# AGENT 2: Competitor Research Agent
competitor_agent = Agent(
    name="Competitor Research Agent",

    instructions="""
    You are the Competitor Research Agent for ProductPilot AI.

    StudyFlow is a fictional student productivity application.

    You MUST call get_competitor_data before performing
    any competitor analysis.

    Your responsibilities:

    1. Compare StudyFlow with competing products.
    2. Identify features missing from StudyFlow.
    3. Identify features commonly offered by competitors.
    4. Compare the provided performance ratings.
    5. Recommend the top 3 competitive improvements.

    IMPORTANT:
    - Use only information from the dataset.
    - All competitors and ratings are fictional.
    - Do not invent market statistics.
    - Do not claim to have performed live internet research.
    - A feature value of 1 means available.
    - A feature value of 0 means unavailable.
    - Performance ratings range from 1 to 5.
    - Basic study tracking does not necessarily include
      detailed daily study-hour tracking.

    Keep your analysis concise and evidence-based.
    """,

    model=model,
    tools=[get_competitor_data]
)

print("Tool 3 created successfully!")
print("Competitor Research Agent initialized!")

Tool 3 created successfully!
Competitor Research Agent initialized!


In [76]:
# Run Competitor Research Agent
result = await Runner.run(
    competitor_agent,
    "Analyze StudyFlow's competitors, identify missing "
    "features, and recommend the top 3 improvements."
)

print("=== PRODUCTPILOT AI ===")
print("COMPETITOR RESEARCH REPORT\n")

print(result.final_output)

print("\n=== TOOL EXECUTION LOG ===")

for item in result.new_items:
    if item.type == "tool_call_item":
        print("Tool called:", item.raw_item.name)

    elif item.type == "tool_call_output_item":
        print("Tool output received")

    else:
        print(type(item).__name__)

=== PRODUCTPILOT AI ===
COMPETITOR RESEARCH REPORT

I have analyzed StudyFlow's competitors based on the provided data. Here's a summary of the analysis and recommendations:

**Competitor Analysis:**

StudyFlow is currently lacking in several features compared to its competitors:

*   **Deadline Reminders:** StudyFlow (0) does not offer deadline reminders, while TaskMaster (1) and EduPlanner (1) do.
*   **Calendar Integration:** StudyFlow (0) does not have a calendar feature, while TaskMaster (1), EduPlanner (1), and FocusMate (1) do.
*   **Dark Mode:** StudyFlow (0) does not offer a dark mode, while TaskMaster (1) and FocusMate (1) do.
*   **Cross-Device Sync:** StudyFlow (0) does not offer cross-device synchronization, while TaskMaster (1) and EduPlanner (1) do.

**Features Commonly Offered by Competitors:**

*   **Study Tracking:** This is available in StudyFlow (1), EduPlanner (1), and FocusMate (1).
*   **Deadline Reminders:** Available in TaskMaster (1) and EduPlanner (1).
*   **

In [ ]:
import pandas as pd

# Fictional engineering estimates for StudyFlow
engineering_data = [
    {
        "feature": "Deadline Reminders",
        "estimated_effort_days": 5,
        "business_value": 9
    },
    {
        "feature": "Calendar",
        "estimated_effort_days": 10,
        "business_value": 8
    },
    {
        "feature": "Performance & Stability",
        "estimated_effort_days": 7,
        "business_value": 9
    },
    {
        "feature": "Dark Mode",
        "estimated_effort_days": 3,
        "business_value": 5
    },
    {
        "feature": "Study Hour Tracking",
        "estimated_effort_days": 6,
        "business_value": 6
    },
    {
        "feature": "Study Progress Dashboard",
        "estimated_effort_days": 8,
        "business_value": 7
    },
    {
        "feature": "Cross-Device Sync",
        "estimated_effort_days": 12,
        "business_value": 8
    }
]

engineering_df = pd.DataFrame(engineering_data)

# Save as CSV
engineering_df.to_csv(
    "engineering_estimates.csv",
    index=False
)

print("Engineering estimates dataset created!")
print("Total features:", len(engineering_df))

display(engineering_df)

Engineering estimates dataset created!
Total features: 7


,feature,estimated_effort_days,business_value
0,Deadline Reminders,5,9
1,Calendar,10,8
2,Performance & Stability,7,9
3,Dark Mode,3,5
4,Study Hour Tracking,6,6
5,Study Progress Dashboard,8,7
6,Cross-Device Sync,12,8


In [80]:
from agents import function_tool
import pandas as pd

@function_tool
def calculate_feature_priority() -> str:
    """
    Calculate feature priority scores using customer demand,
    competitor gaps, business value, and implementation ease.
    """

    try:
        # Load datasets
        feedback = pd.read_csv("customer_feedback.csv")
        competitors = pd.read_csv("competitor_data.csv")
        engineering = pd.read_csv("engineering_estimates.csv")

        # Calculate customer demand
        demand_counts = feedback["category"].value_counts()

        max_demand = demand_counts.max()

        # Map features to competitor dataset columns
        feature_mapping = {
            "Deadline Reminders": "deadline_reminders",
            "Calendar": "calendar",
            "Performance & Stability": "performance_rating",
            "Dark Mode": "dark_mode",
            "Study Hour Tracking": "study_tracking",
            "Study Progress Dashboard": None,
            "Cross-Device Sync": "cross_device_sync"
        }

        results = []

        max_effort = engineering["estimated_effort_days"].max()

        for _, row in engineering.iterrows():

            feature = row["feature"]
            effort = row["estimated_effort_days"]
            business_value = row["business_value"]

            # 1. Customer demand score
            demand = demand_counts.get(feature, 0)
            demand_score = (demand / max_demand) * 10

            # 2. Competitor gap score
            column = feature_mapping.get(feature)

            if column is None:
                competitor_score = 0

            elif column == "performance_rating":
                studyflow_rating = competitors.loc[
                    competitors["product"] == "StudyFlow",
                    "performance_rating"
                ].iloc[0]

                other_ratings = competitors.loc[
                    competitors["product"] != "StudyFlow",
                    "performance_rating"
                ]

                competitor_score = max(
                    0,
                    (other_ratings.mean() - studyflow_rating) / 5 * 10
                )

            else:
                studyflow_value = competitors.loc[
                    competitors["product"] == "StudyFlow",
                    column
                ].iloc[0]

                competitor_values = competitors.loc[
                    competitors["product"] != "StudyFlow",
                    column
                ]

                if studyflow_value == 0:
                    competitor_score = competitor_values.mean() * 10
                else:
                    competitor_score = 0

            # 3. Implementation ease
            ease_score = (1 - effort / max_effort) * 10

            # 4. Weighted priority score
            priority_score = (
                0.40 * demand_score
                + 0.25 * competitor_score
                + 0.20 * business_value
                + 0.15 * ease_score
            )

            results.append({
                "Feature": feature,
                "Demand": demand,
                "Effort_Days": effort,
                "Priority_Score": round(priority_score, 2)
            })

        # Convert results into DataFrame
        ranking = pd.DataFrame(results)

        # Sort by highest priority
        ranking = ranking.sort_values(
            by="Priority_Score",
            ascending=False
        )

        return ranking.to_string(index=False)

    except Exception as e:
        return f"Error calculating priorities: {str(e)}"


print("Tool 4 created successfully!")

Tool 4 created successfully!


In [81]:
from agents import Agent, Runner

# AGENT 3: Feature Prioritization Agent
prioritization_agent = Agent(
    name="Feature Prioritization Agent",

    instructions="""
    You are the Feature Prioritization Agent for ProductPilot AI.

    Your responsibility is to recommend which StudyFlow
    product improvements should be developed first.

    IMPORTANT:
    You MUST call the calculate_feature_priority tool
    before making recommendations.

    The tool calculates priority scores using:

    - Customer Demand: 40%
    - Competitor Gap: 25%
    - Business Value: 20%
    - Implementation Ease: 15%

    Your responsibilities:

    1. Retrieve the calculated feature rankings.
    2. Present all features in descending priority order.
    3. Explain why the top 3 features rank highly.
    4. Identify any important trade-offs.
    5. Recommend which feature to develop first.

    RULES:
    - Never invent priority scores.
    - Use the exact scores returned by the tool.
    - Do not change the calculated rankings.
    - Explain recommendations in beginner-friendly language.
    - Treat engineering estimates and business values
      as fictional planning assumptions.
    - Do not claim that a feature has been approved.

    Keep the report concise and professional.
    """,

    model=model,
    tools=[calculate_feature_priority]
)

print("Feature Prioritization Agent initialized!")

Feature Prioritization Agent initialized!


In [82]:
# Execute Feature Prioritization Agent
result = await Runner.run(
    prioritization_agent,
    "Calculate StudyFlow feature priorities, rank all "
    "features, and recommend the best improvement to "
    "develop first."
)

print("=== PRODUCTPILOT AI ===")
print("FEATURE PRIORITIZATION REPORT\n")

print(result.final_output)

print("\n=== TOOL EXECUTION LOG ===")

for item in result.new_items:

    if item.type == "tool_call_item":
        print("Tool called:", item.raw_item.name)

    elif item.type == "tool_call_output_item":
        print("Tool output received")

    else:
        print(type(item).__name__)

=== PRODUCTPILOT AI ===
FEATURE PRIORITIZATION REPORT

Here are the StudyFlow feature priorities, ranked in descending order:

1.  **Deadline Reminders**: Priority Score 8.34
2.  **Calendar**: Priority Score 6.75
3.  **Performance & Stability**: Priority Score 5.49
4.  **Dark Mode**: Priority Score 4.59
5.  **Cross-Device Sync**: Priority Score 4.07
6.  **Study Hour Tracking**: Priority Score 2.75
7.  **Study Progress Dashboard**: Priority Score 2.70

**Explanation of Top 3 Features:**

1.  **Deadline Reminders**: This feature has the highest priority because it scores very well in Customer Demand. This means many users want this functionality, making it a crucial improvement.
2.  **Calendar**: The Calendar feature also ranks highly. While customer demand isn't as high as Deadline Reminders, it still has a solid score, indicating it's a valuable addition that users are looking for.
3.  **Performance & Stability**: This feature is important for keeping the app running smoothly. Although

In [ ]:
# STEP 14: Human Approval Checkpoint

recommended_feature = "Deadline Reminders"

print("=== PRODUCTPILOT AI ===")
print("HUMAN APPROVAL CHECKPOINT\n")

print("Recommended Feature:", recommended_feature)
print("Priority Score: 8.34")
print("\nAvailable decisions:")
print("1. Approve")
print("2. Reject")
print("3. Request Changes")

decision = input("\nEnter your decision (1/2/3): ").strip()

approved_feature = None
approval_status = None

if decision == "1":
    approved_feature = recommended_feature
    approval_status = "APPROVED"

    print("\nFeature approved!")
    print("Approved Feature:", approved_feature)

elif decision == "2":
    approval_status = "REJECTED"

    print("\nFeature rejected.")
    print("No PRD will be generated.")

elif decision == "3":
    approval_status = "CHANGES_REQUESTED"

    feedback = input("Describe the changes required: ").strip()

    print("\nChanges requested:")
    print(feedback)
    print("Feature requires review before approval.")

else:
    approval_status = "INVALID"

    print("\nInvalid selection. Please rerun the cell.")

=== PRODUCTPILOT AI ===
HUMAN APPROVAL CHECKPOINT

Recommended Feature: Deadline Reminders
Priority Score: 8.34

Available decisions:
1. Approve
2. Reject
3. Request Changes

Enter your decision (1/2/3): 1

Feature approved!
Approved Feature: Deadline Reminders


In [35]:
from agents import Agent, function_tool
from pathlib import Path

# TOOL 5: Save Product Requirements Document
@function_tool
def save_prd(feature_name: str, prd_content: str) -> str:
    """
    Save a Product Requirements Document as a Markdown file.

    Args:
        feature_name: Name of the approved product feature.
        prd_content: Complete PRD text in Markdown format.

    Returns:
        Confirmation message with the saved file path.
    """
    # Verify approval using the saved workflow state
    import json

    state_path = Path("productpilot_outputs/workflow_state.json")

    if not state_path.is_file():
        return "Error: Workflow state is missing. PRD generation blocked."

    try:
        state = json.loads(state_path.read_text(encoding="utf-8"))
        approval = state.get("approval", {})

        if approval.get("status") != "APPROVED":
            return "Error: Human approval is required before saving a PRD."

        if approval.get("approved_feature") != feature_name:
            return "Error: This feature is not the approved feature."

    except (OSError, ValueError, TypeError):
        return "Error: Cannot verify workflow approval."

    try:
        # Create a folder for generated documents
        output_dir = Path("productpilot_outputs")
        output_dir.mkdir(exist_ok=True)

        # Create a safe filename
        safe_name = "".join(
            char if char.isalnum() else "_"
            for char in feature_name.lower()
        ).strip("_")

        if not safe_name or not prd_content.strip():
            return "Error: Feature name and PRD content cannot be empty."

        file_path = output_dir / f"{safe_name}_prd.md"

        # Save the PRD
        file_path.write_text(
            prd_content,
            encoding="utf-8"
        )

        return f"PRD saved successfully: {file_path}"

    except Exception as e:
        return f"Error saving PRD: {str(e)}"


# AGENT 4: PRD Writer
prd_agent = Agent(
    name="PRD Writer Agent",

    instructions="""
    You are the PRD Writer Agent for ProductPilot AI.

    Your responsibility is to write professional Product
    Requirements Documents for approved StudyFlow features.

    StudyFlow is a fictional student productivity application.

    IMPORTANT:
    Only create a PRD when the user explicitly provides
    an approved feature and confirms its approval status.

    Your PRD must contain:

    1. Feature Title
    2. Problem Statement
    3. Product Objective
    4. Target Users
    5. User Stories
    6. Functional Requirements
    7. Nonfunctional Requirements
    8. Acceptance Criteria
    9. Success Metrics
    10. Assumptions and Risks

    Write specific, practical requirements.

    Clearly label proposed targets and assumptions.
    Do not invent verified customer statistics.

    After generating the PRD, you MUST call the
    save_prd tool to save the complete document.

    Return a concise summary and the saved file path.
    """,

    model=model,
    tools=[save_prd]
)

print("Tool 5 created successfully!")
print("PRD Writer Agent initialized!")

Tool 5 created successfully!
PRD Writer Agent initialized!


In [ ]:
# Ensure the feature was approved
if approval_status == "APPROVED" and approved_feature:

    result = await Runner.run(
        prd_agent,
        f"""
        Generate a complete Product Requirements Document.

        Product: StudyFlow
        Approved Feature: {approved_feature}
        Approval Status: {approval_status}

        Verified customer demand:
        5 out of 15 feedback records requested deadline reminders.

        Engineering estimate:
        5 engineering workdays.

        The feature should help students avoid missing
        assignment deadlines.

        Generate the PRD and save it using your tool.
        """
    )

    print("=== PRODUCTPILOT AI ===")
    print("PRD GENERATION REPORT\n")
    print(result.final_output)

    print("\n=== TOOL EXECUTION LOG ===")

    for item in result.new_items:

        if item.type == "tool_call_item":
            print("Tool called:", item.raw_item.name)

        elif item.type == "tool_call_output_item":
            print("Tool output received")

        else:
            print(type(item).__name__)

else:
    print("PRD generation blocked: Feature not approved.")

=== PRODUCTPILOT AI ===
PRD GENERATION REPORT

The Product Requirements Document for the "Deadline Reminders" feature has been generated and saved successfully.

**Summary:** The PRD outlines the "Deadline Reminders" feature for StudyFlow, aiming to help students avoid missing assignment deadlines by providing timely, configurable in-app notifications. It includes user stories, functional and non-functional requirements, acceptance criteria, success metrics, and a discussion of assumptions and risks.

**File Path:** `productpilot_outputs/deadline_reminders_prd.md`

=== TOOL EXECUTION LOG ===
MessageOutputItem
Tool called: save_prd
Tool output received
MessageOutputItem


In [43]:
from agents import function_tool
from pathlib import Path

# TOOL 6: Read an approved feature's PRD
@function_tool
def read_approved_prd(feature_name: str) -> str:
    """
    Read the saved Product Requirements Document
    for an approved StudyFlow feature.

    Args:
        feature_name: Name of the approved feature.

    Returns:
        The PRD content or an error message.
    """

    try:
        # Check approval before accessing the document
        if approval_status != "APPROVED":
            return "Error: Feature has not been approved."

        if feature_name != approved_feature:
            return "Error: This feature is not the approved feature."

        # Match the filename created by save_prd()
        safe_name = "".join(
            char if char.isalnum() else "_"
            for char in feature_name.lower()
        ).strip("_")

        file_path = (
            Path("productpilot_outputs")
            / f"{safe_name}_prd.md"
        )

        if not file_path.exists():
            return f"Error: PRD file not found at {file_path}"

        content = file_path.read_text(encoding="utf-8")

        if not content.strip():
            return "Error: PRD file is empty."

        return content

    except Exception as e:
        return f"Error reading PRD: {str(e)}"


print("Tool 6 created successfully!")

Tool 6 created successfully!


In [ ]:
# Test Tool 6 without using Gemini API

from pathlib import Path

# Check approval
assert approval_status == "APPROVED", "Feature is not approved."
assert approved_feature is not None, "No approved feature found."

# Locate the saved PRD
safe_name = "".join(
    char if char.isalnum() else "_"
    for char in approved_feature.lower()
).strip("_")

prd_path = Path("productpilot_outputs") / f"{safe_name}_prd.md"

# Verify file exists
assert prd_path.exists(), f"PRD file not found: {prd_path}"

# Read PRD
prd_text = prd_path.read_text(encoding="utf-8")

assert prd_text.strip(), "PRD file is empty."

print("=== SAVED PRD CONTENT ===\n")
print(prd_text[:2000])

print("\n✅ Tool 6 file-reading logic verified!")

=== SAVED PRD CONTENT ===

# Product Requirements Document: Deadline Reminders

## 1. Feature Title
Deadline Reminders

## 2. Problem Statement
Students often struggle to keep track of multiple assignment and project deadlines across different courses. This can lead to missed deadlines, increased stress, and a negative impact on academic performance. StudyFlow currently lacks a robust system for proactively notifying students about upcoming deadlines.

## 3. Product Objective
To reduce the number of missed assignment deadlines by providing students with timely and configurable reminders within the StudyFlow application.

## 4. Target Users
All registered students using the StudyFlow application.

## 5. User Stories
* As a student, I want to set deadline reminders for my assignments so that I don't forget them.
* As a student, I want to receive notifications for upcoming deadlines so that I can plan my work accordingly.
* As a student, I want to be able to customize when I receive deadl

In [57]:
from agents import Agent, Runner

sprint_agent = Agent(
    name="Sprint Planner Agent",

    instructions="""You are the Sprint Planner Agent for ProductPilot AI.

    Your job is to convert an approved StudyFlow Product
    Requirements Document into a development plan.

    IMPORTANT:
    - You MUST call read_approved_prd to retrieve the PRD.
    - Never create a sprint plan without reading the PRD.
    - Base the plan on the actual PRD.
    - Do not plan unapproved features.

    Create a complete Markdown report with these sections:

    1. Sprint Goal
    2. Development Tasks
    3. Risks and Mitigation
    4. Definition of Done

    For each development task, include:
    - Task ID and name
    - Specific description
    - Effort estimate in engineering days
    - Dependencies
    - Clear acceptance criteria

    Include implementation, testing and integration tasks.

    Use the total engineering estimate in the PRD as a
    planning constraint. If insufficient, flag the risk.

    Clearly distinguish estimates from verified facts.
    Do not claim implementation is complete.

    IMPORTANT OUTPUT RULES:
    - Write concise, meaningful content.
    - Do not generate tables.
    - Do not add excessive whitespace.
    - Do not repeat sections.
    - Finish all sections before ending the response.
    """,

    model=model,
    tools=[read_approved_prd]
)

print("Sprint Planner Agent initialized!")

Sprint Planner Agent initialized!


In [ ]:
if approval_status == "APPROVED" and approved_feature:

    sprint_result = await Runner.run(
        sprint_agent,
        f"""
        Create a development sprint plan.

        Product: StudyFlow
        Approved Feature: {approved_feature}

        Approval Status: {approval_status}
        Total Engineering Estimate: 5 engineering days.

        Read the approved PRD using your tool before planning.

        Break the work into development tasks with
        estimated effort, dependencies, and acceptance criteria.

        Ensure the total planned effort is approximately
        5 engineering days, or explain why more time is needed.
        """
    )

    print("=== PRODUCTPILOT AI ===")
    print("SPRINT PLANNING REPORT\n")
    print(sprint_result.final_output)

    print("\n=== TOOL EXECUTION LOG ===")

    for item in sprint_result.new_items:

        if item.type == "tool_call_item":
            print("Tool called:", item.raw_item.name)

        elif item.type == "tool_call_output_item":
            print("Tool output received")

        else:
            print(type(item).__name__)

else:
    print("Sprint planning blocked: Feature not approved.")

=== PRODUCTPILOT AI ===
SPRINT PLANNING REPORT

## Sprint Plan: Deadline Reminders

**Product:** StudyFlow
**Feature:** Deadline Reminders
**Total Engineering Estimate:** 5 engineering days

### 1. Sprint Goal

Implement a reliable deadline reminder system within StudyFlow to help students proactively manage their assignments and reduce missed deadlines.

### 2. Development Tasks

| Task ID | Task Name                     | Description                                                                                                                            | Effort Estimate (days) | Dependencies | Acceptance Criteria                                                                                                                                                                                                                                   | Risks and Mitigation                                                                                                                               

In [ ]:
from pathlib import Path

# Ensure the sprint planning result exists
assert sprint_result.final_output, "Sprint plan is empty."

# Ensure the feature is approved
assert approval_status == "APPROVED", "Feature is not approved."

# Create output directory
output_dir = Path("productpilot_outputs")
output_dir.mkdir(exist_ok=True)

# Generate filename
safe_name = "".join(
    char if char.isalnum() else "_"
    for char in approved_feature.lower()
).strip("_")

file_path = output_dir / f"{safe_name}_sprint_plan.md"

# Save sprint plan
file_path.write_text(
    str(sprint_result.final_output),
    encoding="utf-8"
)

print("Sprint plan saved successfully!")
print("File:", file_path)

# Verify the saved file
saved_content = file_path.read_text(encoding="utf-8")

assert saved_content.strip(), "Saved file is empty."

print("Verification: PASSED")
print("Characters saved:", len(saved_content))

Sprint plan saved successfully!
File: productpilot_outputs/deadline_reminders_sprint_plan.md
Verification: PASSED
Characters saved: 10356


In [ ]:
import pandas as pd

# Fictional StudyFlow product analytics data

analytics_data = [
    {
        "week": "Week 1",
        "active_users": 420,
        "tasks_created": 1250,
        "tasks_completed": 875,
        "missed_deadlines": 95,
        "average_study_hours": 2.1,
        "retention_rate": 68
    },
    {
        "week": "Week 2",
        "active_users": 450,
        "tasks_created": 1380,
        "tasks_completed": 980,
        "missed_deadlines": 102,
        "average_study_hours": 2.3,
        "retention_rate": 70
    },
    {
        "week": "Week 3",
        "active_users": 480,
        "tasks_created": 1450,
        "tasks_completed": 1040,
        "missed_deadlines": 110,
        "average_study_hours": 2.4,
        "retention_rate": 72
    },
    {
        "week": "Week 4",
        "active_users": 510,
        "tasks_created": 1560,
        "tasks_completed": 1130,
        "missed_deadlines": 118,
        "average_study_hours": 2.5,
        "retention_rate": 74
    }
]

analytics_df = pd.DataFrame(analytics_data)

# Save dataset
analytics_df.to_csv(
    "product_analytics.csv",
    index=False
)

print("Product analytics dataset created successfully!")
print("Total weeks:", len(analytics_df))

display(analytics_df)

Product analytics dataset created successfully!
Total weeks: 4


,week,active_users,tasks_created,tasks_completed,missed_deadlines,average_study_hours,retention_rate
0,Week 1,420,1250,875,95,2.1,68
1,Week 2,450,1380,980,102,2.3,70
2,Week 3,480,1450,1040,110,2.4,72
3,Week 4,510,1560,1130,118,2.5,74


In [62]:
from agents import function_tool
import pandas as pd

# TOOL 7: Calculate Product Metrics

@function_tool
def calculate_product_metrics() -> str:
    """
    Calculate verified StudyFlow product metrics
    using the synthetic analytics dataset.
    """

    try:
        df = pd.read_csv("product_analytics.csv")

        required_columns = [
            "week",
            "active_users",
            "tasks_created",
            "tasks_completed",
            "missed_deadlines",
            "average_study_hours",
            "retention_rate"
        ]

        # Validate dataset
        missing = [
            col for col in required_columns
            if col not in df.columns
        ]

        if missing:
            return f"Error: Missing columns: {missing}"

        if df.empty:
            return "Error: Analytics dataset is empty."

        if df[required_columns].isnull().any().any():
            return "Error: Analytics dataset contains missing values."

        if (df["tasks_created"] <= 0).any():
            return "Error: Tasks created must be positive."

        if (df["active_users"] <= 0).any():
            return "Error: Active users must be positive."

        # Calculate weekly rates
        df["completion_rate"] = (
            df["tasks_completed"]
            / df["tasks_created"]
            * 100
        )

        df["missed_deadline_rate"] = (
            df["missed_deadlines"]
            / df["tasks_created"]
            * 100
        )

        # Calculate overall metrics
        total_tasks = df["tasks_created"].sum()
        total_completed = df["tasks_completed"].sum()
        total_missed = df["missed_deadlines"].sum()

        overall_completion_rate = (
            total_completed / total_tasks * 100
        )

        overall_missed_rate = (
            total_missed / total_tasks * 100
        )

        # Active user growth: Week 1 to Week 4
        first_week_users = df["active_users"].iloc[0]
        last_week_users = df["active_users"].iloc[-1]

        user_growth = (
            (last_week_users - first_week_users)
            / first_week_users
            * 100
        )

        # Average weekly metrics
        avg_retention = df["retention_rate"].mean()
        avg_study_hours = df["average_study_hours"].mean()

        # Format weekly report
        weekly_report = df[
            [
                "week",
                "active_users",
                "completion_rate",
                "missed_deadline_rate",
                "retention_rate"
            ]
        ].round(2).to_string(index=False)

        report = f"""
STUDYFLOW PRODUCT ANALYTICS REPORT
(Synthetic Data)

WEEKLY METRICS:
{weekly_report}

OVERALL METRICS:

Total Tasks Created: {total_tasks}
Total Tasks Completed: {total_completed}
Total Missed Deadlines: {total_missed}

Overall Task Completion Rate: {overall_completion_rate:.2f}%
Overall Missed Deadline Rate: {overall_missed_rate:.2f}%

Active User Growth (Week 1 to Week 4): {user_growth:.2f}%

Average Weekly Retention Rate: {avg_retention:.2f}%
Average Reported Study Hours: {avg_study_hours:.2f}

IMPORTANT:
- All data is fictional.
- Rates use tasks created as the denominator.
- Retention is a simple average of weekly percentages,
  not a cohort-based retention calculation.
- Study hours are averaged across the four weekly values.
- These metrics do not establish causal relationships.
"""

        return report

    except Exception as e:
        return f"Error calculating product metrics: {str(e)}"


print("Tool 7 created successfully!")

Tool 7 created successfully!


In [ ]:
# Offline verification of analytics calculations

df = pd.read_csv("product_analytics.csv")

total_tasks = df["tasks_created"].sum()
total_completed = df["tasks_completed"].sum()
total_missed = df["missed_deadlines"].sum()

completion_rate = total_completed / total_tasks * 100
missed_rate = total_missed / total_tasks * 100

user_growth = (
    (df["active_users"].iloc[-1] - df["active_users"].iloc[0])
    / df["active_users"].iloc[0]
    * 100
)

print("=== ANALYTICS VERIFICATION ===")

print("Total Tasks Created:", total_tasks)
print("Total Tasks Completed:", total_completed)
print("Total Missed Deadlines:", total_missed)

print(f"Completion Rate: {completion_rate:.2f}%")
print(f"Missed Deadline Rate: {missed_rate:.2f}%")
print(f"Active User Growth: {user_growth:.2f}%")

print("\nVerification completed!")

=== ANALYTICS VERIFICATION ===
Total Tasks Created: 5640
Total Tasks Completed: 4025
Total Missed Deadlines: 425
Completion Rate: 71.37%
Missed Deadline Rate: 7.54%
Active User Growth: 21.43%

Verification completed!


In [63]:
from agents import Agent, Runner

# AGENT 6: Product Analytics Agent

analytics_agent = Agent(
    name="Product Analytics Agent",

    instructions="""
    You are the Product Analytics Agent for ProductPilot AI.

    ProductPilot AI is an intelligent multi-agent product
    management assistant.

    StudyFlow is a fictional student productivity application.

    Your responsibility is to analyze product usage metrics
    and provide data-driven product recommendations.

    IMPORTANT:
    You MUST call calculate_product_metrics before
    generating your analytics report.

    YOUR RESPONSIBILITIES:

    1. Present the verified product metrics.
    2. Explain trends in active users, task completion,
       missed deadlines, and retention.
    3. Identify potential product problems.
    4. Recommend three important KPIs to monitor.
    5. Explain how these KPIs could help evaluate
       the Deadline Reminders feature.
    6. Suggest realistic next steps for product evaluation.

    RULES:

    - Never invent numerical metrics.
    - Use exact values returned by the tool.
    - Clearly state that the dataset is synthetic.
    - Do not confuse absolute counts with rates.
    - Do not claim that Deadline Reminders has launched.
    - Do not claim that the feature has improved performance.
    - Treat proposed KPI targets as hypothetical.
    - Do not make unsupported causal claims.

    Present a concise, professional Markdown report.
    """,

    model=model,
    tools=[calculate_product_metrics]
)

print("Product Analytics Agent initialized!")

Product Analytics Agent initialized!


In [ ]:
# Test Product Analytics Agent

analytics_result = await Runner.run(
    analytics_agent,
    """
    Analyze StudyFlow's product performance.

    Use the analytics calculation tool to retrieve
    verified metrics.

    Explain the key trends and recommend three KPIs
    that should be monitored after launching the
    Deadline Reminders feature.

    Remember that Deadline Reminders has not launched yet.
    """
)

print("=== PRODUCTPILOT AI ===")
print("PRODUCT ANALYTICS REPORT\n")

print(analytics_result.final_output)

print("\n=== TOOL EXECUTION LOG ===")

for item in analytics_result.new_items:

    if item.type == "tool_call_item":
        print("Tool called:", item.raw_item.name)

    elif item.type == "tool_call_output_item":
        print("Tool output received")

    else:
        print(type(item).__name__)

=== PRODUCTPILOT AI ===
PRODUCT ANALYTICS REPORT

I have analyzed the synthetic product performance data for StudyFlow. Here's a report of the findings and recommendations:

## StudyFlow Product Analytics Report

This report is based on a synthetic dataset and outlines key product metrics, trends, and recommendations for the upcoming Deadline Reminders feature.

### Verified Product Metrics

The following metrics were calculated using the `calculate_product_metrics` tool:

*   **Weekly Metrics:**
    *   Week 1: 420 active users, 70.00% completion rate, 7.60% missed deadline rate, 68% retention rate
    *   Week 2: 450 active users, 71.01% completion rate, 7.39% missed deadline rate, 70% retention rate
    *   Week 3: 480 active users, 71.72% completion rate, 7.59% missed deadline rate, 72% retention rate
    *   Week 4: 510 active users, 72.44% completion rate, 7.56% missed deadline rate, 74% retention rate
*   **Overall Metrics:**
    *   Total Tasks Created: 5640
    *   Total Tasks

In [ ]:
from pathlib import Path

# Create output folder if needed
output_dir = Path("productpilot_outputs")
output_dir.mkdir(exist_ok=True)

# Check that analytics output exists
assert analytics_result.final_output, "Analytics report is empty."

# Save analytics report
analytics_path = output_dir / "studyflow_analytics_report.md"

analytics_path.write_text(
    str(analytics_result.final_output),
    encoding="utf-8"
)

# Verify saved content
saved_report = analytics_path.read_text(encoding="utf-8")

assert saved_report.strip(), "Saved analytics report is empty."

print("Analytics report saved successfully!")
print("File:", analytics_path)
print("Characters saved:", len(saved_report))

Analytics report saved successfully!
File: productpilot_outputs/studyflow_analytics_report.md
Characters saved: 4414


In [ ]:
from agents import Agent, Runner

# Create a handoff-enabled version of Feedback Analysis Agent

feedback_handoff_agent = Agent(
    name="Feedback Analysis Agent",

    instructions="""
    You are the Feedback Analysis Agent for ProductPilot AI.

    Your responsibilities:

    1. Analyze StudyFlow customer feedback.
    2. Identify the most requested improvements.
    3. Use the available tools to obtain verified feedback counts.
    4. Summarize the main customer problems.

    IMPORTANT:

    You MUST call:
    - read_customer_feedback
    - get_feedback_statistics

    After completing the feedback analysis,
    you MUST hand off control to the
    Competitor Research Agent.

    The Competitor Research Agent will examine
    competitor features and identify product gaps.

    Do not produce a final answer before handing off.

    StudyFlow and its competitors are fictional.
    """,

    model=model,

    tools=[
        read_customer_feedback,
        get_feedback_statistics
    ],

    handoffs=[competitor_agent]
)

print("Feedback → Competitor handoff configured!")

Feedback → Competitor handoff configured!


In [ ]:
from agents import Runner

handoff_result = await Runner.run(
    feedback_handoff_agent,
    """
    Analyze StudyFlow's customer feedback.

    Identify the most requested features using
    verified statistics.

    Then transfer control to the Competitor
    Research Agent to analyze competitive gaps.

    Provide a final summary of the findings.
    """
)

print("=== PRODUCTPILOT AI ===")
print("MULTI-AGENT HANDOFF TEST\n")

print(handoff_result.final_output)

print("\n=== EXECUTION LOG ===")

for item in handoff_result.new_items:

    print(type(item).__name__)

    if item.type == "tool_call_item":
        print("Tool called:", item.raw_item.name)

    elif item.type == "handoff_call_item":
        print("Agent handoff initiated!")

    elif item.type == "handoff_output_item":
        print("Agent handoff completed!")

=== PRODUCTPILOT AI ===
MULTI-AGENT HANDOFF TEST

Here's a competitive analysis of StudyFlow:

**Features Missing from StudyFlow:**

*   Deadline Reminders: StudyFlow does not offer this feature (0), while TaskMaster (1) and EduPlanner (1) do.
*   Calendar: StudyFlow lacks a calendar feature (0), which is present in TaskMaster (1), EduPlanner (1), and FocusMate (1).
*   Dark Mode: StudyFlow does not have a dark mode (0), but TaskMaster (1) and FocusMate (1) offer it.
*   Cross-Device Sync: StudyFlow does not support cross-device sync (0), while TaskMaster (1) and EduPlanner (1) do.

**Features Commonly Offered by Competitors:**

*   Deadline Reminders: Offered by TaskMaster and EduPlanner.
*   Calendar: Offered by TaskMaster, EduPlanner, and FocusMate.
*   Dark Mode: Offered by TaskMaster and FocusMate.
*   Study Tracking: Offered by StudyFlow, EduPlanner, and FocusMate.
*   Cross-Device Sync: Offered by TaskMaster and EduPlanner.

**Performance Ratings Comparison:**

*   StudyFlow: 3


In [89]:
from agents import Agent, Runner

# AGENT 2: Competitor Research with handoff
competitor_handoff_agent = Agent(
    name="Competitor Research Agent",

    instructions="""
    You are the Competitor Research Agent for ProductPilot AI.

    Your responsibilities:

    1. Analyze StudyFlow's fictional competitors.
    2. Identify features missing from StudyFlow.
    3. Compare competitor capabilities.
    4. Summarize competitive gaps.

    IMPORTANT:

    You MUST call get_competitor_data.

    After completing your analysis, you MUST
    hand off control to the Feature Prioritization Agent.

    Do not produce the final report yourself.

    The next agent will calculate feature priorities
    using verified Python calculations.

    All products and competitor data are fictional.
    """,

    model=model,

    tools=[get_competitor_data],

    handoffs=[prioritization_agent]
)


# AGENT 1: Feedback Analysis with handoff
feedback_handoff_agent = Agent(
    name="Feedback Analysis Agent",

    instructions="""
    You are the Feedback Analysis Agent for ProductPilot AI.

    Your responsibilities:

    1. Analyze StudyFlow customer feedback.
    2. Identify frequently requested improvements.
    3. Use Python tools for verified feedback statistics.

    You MUST call:
    - read_customer_feedback
    - get_feedback_statistics

    After analyzing feedback, you MUST hand off
    control to the Competitor Research Agent.

    Do not generate the final report yourself.

    StudyFlow is a fictional application.
    """,

    model=model,

    tools=[
        read_customer_feedback,
        get_feedback_statistics
    ],

    handoffs=[competitor_handoff_agent]
)

print("Three-agent workflow configured successfully!")

Three-agent workflow configured successfully!


In [ ]:
three_agent_result = await Runner.run(
    feedback_handoff_agent,
    """
    ProductPilot AI workflow:

    1. Analyze StudyFlow customer feedback.
    2. Analyze fictional competitor products.
    3. Calculate feature priorities.
    4. Recommend the highest-priority improvement.

    Each agent must use its assigned tools.

    Use SDK handoffs to transfer control
    between the specialized agents.

    Present the final feature rankings.
    """
)

print("=== PRODUCTPILOT AI ===")
print("THREE-AGENT WORKFLOW REPORT\n")

print(three_agent_result.final_output)

print("\n=== EXECUTION LOG ===")

for item in three_agent_result.new_items:

    if item.type == "tool_call_item":
        print("Tool called:", item.raw_item.name)

    elif item.type == "tool_call_output_item":
        print("Tool output received")

    elif item.type == "handoff_call_item":
        print("Agent handoff initiated!")

    elif item.type == "handoff_output_item":
        print("Agent handoff completed!")

    else:
        print(type(item).__name__)

=== PRODUCTPILOT AI ===
THREE-AGENT WORKFLOW REPORT

Here are the StudyFlow feature rankings, from highest to lowest priority:

1.  **Deadline Reminders** (Score: 8.34)
2.  **Calendar** (Score: 6.75)
3.  **Performance & Stability** (Score: 5.49)
4.  **Dark Mode** (Score: 4.59)
5.  **Cross-Device Sync** (Score: 4.07)
6.  **Study Hour Tracking** (Score: 2.75)
7.  **Study Progress Dashboard** (Score: 2.70)

**Top 3 Features:**

*   **Deadline Reminders:** This feature is ranked highest because customers are asking for it the most (high Customer Demand). It also doesn't require a huge amount of effort to build, making it a quick win.
*   **Calendar:** This feature also has strong Customer Demand and is seen as important when compared to competitors. While it takes a bit longer to build than reminders, it's still a valuable addition.
*   **Performance & Stability:** Even though customer demand isn't as high as for reminders or a calendar, improving performance is crucial. Competitors are do

In [ ]:
import pandas as pd

# Get the latest feature rankings from our Python tool's data
feedback_df = pd.read_csv("customer_feedback.csv")
engineering_df = pd.read_csv("engineering_estimates.csv")

# Our latest verified ranking from the tested workflow
ranked_features = [
    "Deadline Reminders",
    "Calendar",
    "Performance & Stability",
    "Dark Mode",
    "Cross-Device Sync",
    "Study Hour Tracking",
    "Study Progress Dashboard"
]

print("=== PRODUCTPILOT AI ===")
print("HUMAN APPROVAL CHECKPOINT\n")

print("Available Features:\n")

for i, feature in enumerate(ranked_features, start=1):
    print(f"{i}. {feature}")

print("\nAvailable Decisions:")
print("1. Approve Recommended Feature")
print("2. Approve Another Feature")
print("3. Reject Recommendation")
print("4. Request Changes")

decision = input("\nEnter decision (1/2/3/4): ").strip()

approved_feature = None
approval_status = None
review_feedback = None

if decision == "1":

    approved_feature = ranked_features[0]
    approval_status = "APPROVED"

    print("\nApproved:", approved_feature)

elif decision == "2":

    choice = input("Enter feature number (1-7): ").strip()

    if choice.isdigit() and 1 <= int(choice) <= len(ranked_features):

        approved_feature = ranked_features[int(choice) - 1]
        approval_status = "APPROVED"

        print("\nApproved:", approved_feature)

    else:
        approval_status = "INVALID"
        print("\nInvalid feature selection.")

elif decision == "3":

    approval_status = "REJECTED"

    print("\nRecommendation rejected.")
    print("PRD generation is blocked.")

elif decision == "4":

    approval_status = "CHANGES_REQUESTED"

    review_feedback = input(
        "\nDescribe the changes you want: "
    ).strip()

    print("\nChanges requested:", review_feedback)
    print("Feature must be reviewed again before approval.")

else:

    approval_status = "INVALID"
    print("\nInvalid decision.")


print("\n=== APPROVAL SUMMARY ===")
print("Status:", approval_status)
print("Approved Feature:", approved_feature)

=== PRODUCTPILOT AI ===
HUMAN APPROVAL CHECKPOINT

Available Features:

1. Deadline Reminders
2. Calendar
3. Performance & Stability
4. Dark Mode
5. Cross-Device Sync
6. Study Hour Tracking
7. Study Progress Dashboard

Available Decisions:
1. Approve Recommended Feature
2. Approve Another Feature
3. Reject Recommendation
4. Request Changes

Enter decision (1/2/3/4): 1

Approved: Deadline Reminders

=== APPROVAL SUMMARY ===
Status: APPROVED
Approved Feature: Deadline Reminders


In [64]:
from agents import Agent, Runner

# AGENT 6: Analytics Agent (final agent in this chain)
analytics_handoff_agent = Agent(
    name="Product Analytics Agent",

    instructions="""
    You are the Product Analytics Agent for ProductPilot AI.

    You are the final agent in the post-approval workflow.

    You MUST call calculate_product_metrics.

    Analyze StudyFlow's synthetic product analytics.

    Explain:
    1. Verified product metrics
    2. Important trends
    3. Three KPIs to monitor for the approved feature
    4. Recommended next steps

    IMPORTANT:
    - Do not invent numerical values.
    - Do not claim the feature has launched.
    - Clearly label all data as synthetic.
    - Do not make unsupported causal claims.

    Provide the final analytics summary.
    """,

    model=model,
    tools=[calculate_product_metrics]
)


# AGENT 5: Sprint Planner with handoff
sprint_handoff_agent = Agent(
    name="Sprint Planner Agent",

    instructions="""
    You are the Sprint Planner Agent for ProductPilot AI.

    You MUST call read_approved_prd to retrieve
    the approved feature's saved PRD.

    Use the PRD to create a development plan containing:
    - Sprint goal
    - Development tasks
    - Effort estimates
    - Dependencies
    - Acceptance criteria
    - Risks and mitigations
    - Definition of Done

    Use the engineering estimate provided in the request.

    IMPORTANT:
    You MUST include testing and integration tasks.
    Flag unrealistic estimates instead of inventing certainty.

    After completing the sprint plan, hand off control
    to the Product Analytics Agent.

    Do not generate the final response yourself.
    """,

    model=model,
    tools=[read_approved_prd],
    handoffs=[analytics_handoff_agent]
)


# AGENT 4: PRD Writer with handoff
prd_handoff_agent = Agent(
    name="PRD Writer Agent",

    instructions="""
    You are the Product Analytics Agent for ProductPilot AI.

    You are the final agent in the post-approval workflow.

    The currently approved StudyFlow feature is Calendar.

    You MUST call calculate_product_metrics.

    Analyze StudyFlow's synthetic product analytics.

    Explain:
    1. Verified product metrics
    2. Important trends
    3. Three KPIs to monitor for the approved Calendar feature
    4. Recommended next steps for the product team

    IMPORTANT:
    - Do not invent numerical values.
    - Do not claim Calendar has launched.
    - Clearly label all data as synthetic.
    - Do not make unsupported causal claims.
    - Distinguish existing product metrics from proposed
     Calendar success metrics.

    Provide the final analytics summary.
    """,

    model=model,
    tools=[save_prd],
    handoffs=[sprint_handoff_agent]
)

print("Post-approval handoff chain configured!")
print("PRD Writer → Sprint Planner → Product Analytics")

Post-approval handoff chain configured!
PRD Writer → Sprint Planner → Product Analytics


In [ ]:
from agents import Runner
from pathlib import Path

print("=== PRODUCTPILOT AI ===")
print("POST-APPROVAL WORKFLOW\n")

# Enforce human approval before starting
if approval_status != "APPROVED" or not approved_feature:
    raise PermissionError(
        "Workflow blocked: Human approval is required."
    )

# Find the approved feature's engineering estimate
engineering_df = pd.read_csv("engineering_estimates.csv")

matching_rows = engineering_df[
    engineering_df["feature"] == approved_feature
]

if matching_rows.empty:
    raise ValueError(
        f"No engineering estimate found for {approved_feature}"
    )

estimated_days = int(
    matching_rows.iloc[0]["estimated_effort_days"]
)

print("Approval verified!")
print("Approved Feature:", approved_feature)
print("Estimated Effort:", estimated_days, "days")

# Run the connected agents
post_approval_result = await Runner.run(
    prd_handoff_agent,
    f"""
    ProductPilot AI — Approved Product Development Workflow

    Human approval status: {approval_status}
    Approved feature: {approved_feature}
    Engineering estimate: {estimated_days} days

    Product: StudyFlow (fictional student productivity app)

    The PRD Writer must:
    - Generate a complete PRD for the approved feature.
    - Save it using save_prd.
    - Hand off to the Sprint Planner.

    The Sprint Planner must:
    - Read the saved PRD using read_approved_prd.
    - Create a realistic development plan.
    - Hand off to Product Analytics.

    The Product Analytics Agent must:
    - Call calculate_product_metrics.
    - Analyze synthetic product metrics.
    - Recommend three relevant KPIs.

    Use genuine SDK handoffs.
    """
)

print("\n=== FINAL WORKFLOW OUTPUT ===\n")
print(post_approval_result.final_output)

print("\n=== EXECUTION LOG ===\n")

for item in post_approval_result.new_items:

    if item.type == "tool_call_item":
        print("Tool called:", item.raw_item.name)

    elif item.type == "tool_call_output_item":
        print("Tool output received")

    elif item.type == "handoff_call_item":
        print("Agent handoff initiated!")

    elif item.type == "handoff_output_item":
        print("Agent handoff completed!")

    else:
        print(type(item).__name__)

=== PRODUCTPILOT AI ===
POST-APPROVAL WORKFLOW

Approval verified!
Approved Feature: Deadline Reminders
Estimated Effort: 5 days


RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 23h17m23.248889257s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See

In [ ]:
from pathlib import Path

output_dir = Path("productpilot_outputs")

print("=== PRODUCTPILOT AI ===")
print("SAVED OUTPUT FILES\n")

if output_dir.exists():

    files = sorted(output_dir.glob("*"))

    if files:
        for file in files:
            print(f"✅ {file.name}")
            print(f"   Size: {file.stat().st_size} bytes")

    else:
        print("No saved files found.")

else:
    print("Output directory does not exist.")

print("\n=== WORKFLOW STATUS ===")

print("Human Approval:", approval_status)
print("Approved Feature:", approved_feature)

print("\nPre-approval handoffs: VERIFIED")
print("Post-approval handoffs: NOT YET VERIFIED (API quota)")

=== PRODUCTPILOT AI ===
SAVED OUTPUT FILES

✅ deadline_reminders_prd.md
   Size: 3977 bytes
✅ deadline_reminders_sprint_plan.md
   Size: 10356 bytes
✅ studyflow_analytics_report.md
   Size: 4414 bytes

=== WORKFLOW STATUS ===
Human Approval: APPROVED
Approved Feature: Deadline Reminders

Pre-approval handoffs: VERIFIED
Post-approval handoffs: NOT YET VERIFIED (API quota)


In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone

# Create output directory if needed
output_dir = Path("productpilot_outputs")
output_dir.mkdir(exist_ok=True)

# Verified feature rankings from our successful workflow
feature_rankings = [
    {"feature": "Deadline Reminders", "score": 8.34},
    {"feature": "Calendar", "score": 6.75},
    {"feature": "Performance & Stability", "score": 5.49},
    {"feature": "Dark Mode", "score": 4.59},
    {"feature": "Cross-Device Sync", "score": 4.07},
    {"feature": "Study Hour Tracking", "score": 2.75},
    {"feature": "Study Progress Dashboard", "score": 2.70}
]

# Files created during our earlier successful tests
report_files = {
    "prd": output_dir / "deadline_reminders_prd.md",
    "sprint_plan": output_dir / "deadline_reminders_sprint_plan.md",
    "analytics_report": output_dir / "studyflow_analytics_report.md"
}

# Record only files that actually exist
saved_reports = {
    name: str(path)
    for name, path in report_files.items()
    if path.is_file() and path.stat().st_size > 0
}

# Save the workflow's current state
workflow_state = {
    "project": "ProductPilot AI",
    "product": "StudyFlow",
    "data_type": "synthetic",
    "updated_at": datetime.now(timezone.utc).isoformat(),

    "approval": {
        "status": approval_status,
        "approved_feature": approved_feature,
        "review_feedback": globals().get("review_feedback")
    },

    "feature_rankings": feature_rankings,

    "workflow": {
        "pre_approval_handoffs": "VERIFIED",
        "post_approval_handoffs": "PENDING_VERIFICATION",
        "last_error": "Gemini daily free-tier quota exceeded"
    },

    "reports": saved_reports
}

state_path = output_dir / "workflow_state.json"

with open(state_path, "w", encoding="utf-8") as file:
    json.dump(workflow_state, file, indent=4)

print("=== PRODUCTPILOT AI ===")
print("WORKFLOW STATE SAVED\n")

print("File:", state_path)
print("Approval:", workflow_state["approval"]["status"])
print("Feature:", workflow_state["approval"]["approved_feature"])
print("Reports recorded:", len(saved_reports))

print("\nWorkflow state:")
print(json.dumps(workflow_state["workflow"], indent=4))

=== PRODUCTPILOT AI ===
WORKFLOW STATE SAVED

File: productpilot_outputs/workflow_state.json
Approval: APPROVED
Feature: Deadline Reminders
Reports recorded: 3

Workflow state:
{
    "pre_approval_handoffs": "VERIFIED",
    "post_approval_handoffs": "PENDING_VERIFICATION",
    "last_error": "Gemini daily free-tier quota exceeded"
}


In [ ]:
import json
from pathlib import Path

state_path = Path("productpilot_outputs/workflow_state.json")

assert state_path.exists(), "Workflow state file missing!"

with open(state_path, "r", encoding="utf-8") as file:
    restored_state = json.load(file)

assert restored_state["approval"]["status"] == "APPROVED"
assert restored_state["approval"]["approved_feature"] == "Deadline Reminders"
assert len(restored_state["reports"]) == 3

print("✅ Workflow state restored successfully!")
print("Approved feature:", restored_state["approval"]["approved_feature"])
print("Available reports:", list(restored_state["reports"].keys()))

✅ Workflow state restored successfully!
Approved feature: Deadline Reminders
Available reports: ['prd', 'sprint_plan', 'analytics_report']


In [ ]:
import pandas as pd

def generate_dynamic_rankings():

    feedback = pd.read_csv("customer_feedback.csv")
    competitors = pd.read_csv("competitor_data.csv")
    engineering = pd.read_csv("engineering_estimates.csv")

    # Validate input data
    required_feedback = {"category"}
    required_competitors = {
        "product",
        "deadline_reminders",
        "calendar",
        "dark_mode",
        "study_tracking",
        "cross_device_sync",
        "performance_rating"
    }
    required_engineering = {
        "feature",
        "estimated_effort_days",
        "business_value"
    }

    assert required_feedback.issubset(feedback.columns)
    assert required_competitors.issubset(competitors.columns)
    assert required_engineering.issubset(engineering.columns)

    # Customer demand
    demand_counts = feedback["category"].value_counts()

    max_demand = demand_counts.max()
    max_effort = engineering["estimated_effort_days"].max()

    # StudyFlow and its fictional competitors
    studyflow = competitors[
        competitors["product"] == "StudyFlow"
    ].iloc[0]

    other_competitors = competitors[
        competitors["product"] != "StudyFlow"
    ]

    feature_columns = {
        "Deadline Reminders": "deadline_reminders",
        "Calendar": "calendar",
        "Dark Mode": "dark_mode",
        "Study Hour Tracking": "study_tracking",
        "Cross-Device Sync": "cross_device_sync"
    }

    results = []

    for _, row in engineering.iterrows():

        feature = row["feature"]
        effort = row["estimated_effort_days"]
        business_value = row["business_value"]

        # 1. Customer demand: 40%
        demand = demand_counts.get(feature, 0)

        demand_score = (
            demand / max_demand
        ) * 10 if max_demand else 0

        # 2. Competitor gap: 25%
        if feature == "Performance & Stability":

            competitor_gap = (
                other_competitors["performance_rating"].mean()
                - studyflow["performance_rating"]
            ) / 5 * 10

            competitor_gap = max(0, competitor_gap)

        elif feature in feature_columns:

            column = feature_columns[feature]

            if studyflow[column] == 1:
                competitor_gap = 0
            else:
                competitor_gap = (
                    other_competitors[column].mean() * 10
                )

        else:
            # No comparable competitor data available
            competitor_gap = 0

        # 3. Business value: 20%
        business_score = business_value

        # 4. Implementation ease: 15%
        ease_score = (
            1 - effort / max_effort
        ) * 10 if max_effort else 0

        # Weighted priority score
        priority_score = (
            0.40 * demand_score
            + 0.25 * competitor_gap
            + 0.20 * business_score
            + 0.15 * ease_score
        )

        results.append({
            "feature": feature,
            "priority_score": round(priority_score, 2),
            "customer_requests": int(demand),
            "estimated_effort_days": int(effort)
        })

    rankings_df = pd.DataFrame(results)

    rankings_df = rankings_df.sort_values(
        by="priority_score",
        ascending=False
    ).reset_index(drop=True)

    return rankings_df


# Generate rankings directly from CSV files
dynamic_rankings = generate_dynamic_rankings()

print("=== PRODUCTPILOT AI ===")
print("DYNAMIC FEATURE RANKINGS\n")

print(dynamic_rankings.to_string(index=False))

=== PRODUCTPILOT AI ===
DYNAMIC FEATURE RANKINGS

                 feature  priority_score  customer_requests  estimated_effort_days
      Deadline Reminders            8.34                  5                      5
                Calendar            6.75                  3                     10
 Performance & Stability            5.49                  3                      7
               Dark Mode            4.59                  1                      3
       Cross-Device Sync            4.07                  1                     12
     Study Hour Tracking            2.75                  1                      6
Study Progress Dashboard            2.70                  1                      8


In [ ]:
expected_scores = {
    "Deadline Reminders": 8.34,
    "Calendar": 6.75,
    "Performance & Stability": 5.49,
    "Dark Mode": 4.59,
    "Cross-Device Sync": 4.07,
    "Study Hour Tracking": 2.75,
    "Study Progress Dashboard": 2.70
}

print("=== PRIORITY SCORE VALIDATION ===\n")

all_passed = True

for _, row in dynamic_rankings.iterrows():

    feature = row["feature"]
    actual = row["priority_score"]
    expected = expected_scores[feature]

    passed = abs(actual - expected) <= 0.01

    if not passed:
        all_passed = False

    status = "PASS" if passed else "CHECK"

    print(
        f"{status}: {feature} | "
        f"Expected: {expected} | Actual: {actual}"
    )

assert all_passed, "Some scores differ from the previous results!"

print("\n✅ Dynamic ranking validation passed!")

=== PRIORITY SCORE VALIDATION ===

PASS: Deadline Reminders | Expected: 8.34 | Actual: 8.34
PASS: Calendar | Expected: 6.75 | Actual: 6.75
PASS: Performance & Stability | Expected: 5.49 | Actual: 5.49
PASS: Dark Mode | Expected: 4.59 | Actual: 4.59
PASS: Cross-Device Sync | Expected: 4.07 | Actual: 4.07
PASS: Study Hour Tracking | Expected: 2.75 | Actual: 2.75
PASS: Study Progress Dashboard | Expected: 2.7 | Actual: 2.7

✅ Dynamic ranking validation passed!


In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone

# Generate rankings from the latest CSV data
dynamic_rankings = generate_dynamic_rankings()

print("=== PRODUCTPILOT AI ===")
print("DYNAMIC HUMAN APPROVAL\n")

for i, row in dynamic_rankings.iterrows():
    print(
        f"{i+1}. {row['feature']} "
        f"(Score: {row['priority_score']:.2f})"
    )

recommended_feature = dynamic_rankings.iloc[0]["feature"]

print(f"\nAI workflow's top-ranked feature: {recommended_feature}")

print("\nChoose an action:")
print("1. Approve recommended feature")
print("2. Approve another feature")
print("3. Reject recommendation")
print("4. Request changes")

decision = input("\nEnter your choice (1-4): ").strip()

approved_feature = None
review_feedback = None
approval_status = "INVALID"

if decision == "1":
    approved_feature = recommended_feature
    approval_status = "APPROVED"

elif decision == "2":
    choice = input("Enter feature number: ").strip()

    if choice.isdigit():
        index = int(choice) - 1

        if 0 <= index < len(dynamic_rankings):
            approved_feature = dynamic_rankings.iloc[index]["feature"]
            approval_status = "APPROVED"

elif decision == "3":
    approval_status = "REJECTED"

elif decision == "4":
    review_feedback = input(
        "What changes would you like? "
    ).strip()

    if review_feedback:
        approval_status = "CHANGES_REQUESTED"

# Update saved workflow state
state_path = Path("productpilot_outputs/workflow_state.json")

with open(state_path, "r", encoding="utf-8") as file:
    workflow_state = json.load(file)

workflow_state["approval"] = {
    "status": approval_status,
    "approved_feature": approved_feature,
    "review_feedback": review_feedback
}

workflow_state["feature_rankings"] = [
    {
        "feature": row["feature"],
        "score": float(row["priority_score"])
    }
    for _, row in dynamic_rankings.iterrows()
]

workflow_state["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

with open(state_path, "w", encoding="utf-8") as file:
    json.dump(workflow_state, file, indent=4)

print("\n=== APPROVAL RESULT ===")
print("Status:", approval_status)
print("Approved Feature:", approved_feature)
print("Review Feedback:", review_feedback)
print("Checkpoint updated:", state_path)

=== PRODUCTPILOT AI ===
DYNAMIC HUMAN APPROVAL

1. Deadline Reminders (Score: 8.34)
2. Calendar (Score: 6.75)
3. Performance & Stability (Score: 5.49)
4. Dark Mode (Score: 4.59)
5. Cross-Device Sync (Score: 4.07)
6. Study Hour Tracking (Score: 2.75)
7. Study Progress Dashboard (Score: 2.70)

AI workflow's top-ranked feature: Deadline Reminders

Choose an action:
1. Approve recommended feature
2. Approve another feature
3. Reject recommendation
4. Request changes

Enter your choice (1-4): 1

=== APPROVAL RESULT ===
Status: APPROVED
Approved Feature: Deadline Reminders
Review Feedback: None
Checkpoint updated: productpilot_outputs/workflow_state.json


In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone

state_path = Path("productpilot_outputs/workflow_state.json")

with open(state_path, "r", encoding="utf-8") as file:
    workflow_state = json.load(file)

print("=== PRODUCTPILOT AI ===")
print("HUMAN REVISION SYSTEM\n")

print("Current recommendation:")
print(workflow_state["feature_rankings"][0]["feature"])

print("\nEnter a revision request.")
print("Example: Reconsider Calendar because scheduling is important.")
print("Type CANCEL to keep the current approval unchanged.\n")

revision_request = input("Your feedback: ").strip()

if revision_request.upper() == "CANCEL":

    print("\nRevision cancelled.")
    print("Existing approval remains unchanged.")

elif not revision_request:

    print("\nEmpty request. No changes made.")

else:

    # Revoke approval until the revision is reviewed
    workflow_state["approval"] = {
        "status": "CHANGES_REQUESTED",
        "approved_feature": None,
        "review_feedback": revision_request
    }

    workflow_state["workflow"]["revision_status"] = "PENDING_AI_REVIEW"

    workflow_state["updated_at"] = datetime.now(
        timezone.utc
    ).isoformat()

    with open(state_path, "w", encoding="utf-8") as file:
        json.dump(workflow_state, file, indent=4)

    # Keep notebook variables synchronized
    approval_status = "CHANGES_REQUESTED"
    approved_feature = None
    review_feedback = revision_request

    print("\nRevision request saved!")
    print("Approval Status:", approval_status)
    print("Revision Status: PENDING_AI_REVIEW")
    print("PRD Generation: BLOCKED")

=== PRODUCTPILOT AI ===
HUMAN REVISION SYSTEM

Current recommendation:
Deadline Reminders

Enter a revision request.
Example: Reconsider Calendar because scheduling is important.
Type CANCEL to keep the current approval unchanged.

Your feedback: Reconsider Calendar because students need better scheduling.

Revision request saved!
Approval Status: CHANGES_REQUESTED
Revision Status: PENDING_AI_REVIEW
PRD Generation: BLOCKED


In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone

state_path = Path("productpilot_outputs/workflow_state.json")

with open(state_path, "r", encoding="utf-8") as file:
    workflow_state = json.load(file)

# Check that a human revision request actually exists
approval = workflow_state["approval"]

if approval["status"] != "CHANGES_REQUESTED":
    raise PermissionError(
        "No pending human revision request."
    )

review_feedback = approval.get("review_feedback")

if not review_feedback:
    raise ValueError(
        "Revision feedback is missing."
    )

# Retrieve verified rankings from saved state
rankings = workflow_state["feature_rankings"]

ranking_text = "\n".join(
    f"{i}. {item['feature']}: {item['score']:.2f}"
    for i, item in enumerate(rankings, start=1)
)

# Prepare the request for the AI agent
revision_prompt = f"""
You are reviewing a human product manager's feedback.

Product: StudyFlow (fictional student productivity app)

CURRENT VERIFIED FEATURE RANKINGS:
{ranking_text}

HUMAN REVISION REQUEST:
{review_feedback}

YOUR TASK:

1. Carefully evaluate the human's concern.
2. Compare the relevant features.
3. Explain the trade-offs.
4. Decide whether the original recommendation
   should remain unchanged or be revised.
5. Clearly explain your reasoning.

IMPORTANT RULES:

- Treat the supplied rankings as verified calculations.
- Do not invent new priority scores.
- Do not silently modify the scoring formula.
- A human may reasonably choose a lower-ranked feature.
- Clearly distinguish calculated ranking from product judgment.
- Do not generate a PRD.
- Do not claim that any feature has been approved.
"""

# Save request so it survives notebook interruptions
revision_prompt_path = Path(
    "productpilot_outputs/revision_request.txt"
)

revision_prompt_path.write_text(
    revision_prompt,
    encoding="utf-8"
)

workflow_state["workflow"]["revision_status"] = "READY_FOR_AI_REVIEW"

workflow_state["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

with open(state_path, "w", encoding="utf-8") as file:
    json.dump(workflow_state, file, indent=4)

print("=== PRODUCTPILOT AI ===")
print("AI REVISION REQUEST PREPARED\n")

print("Human feedback:", review_feedback)
print("Original recommendation:", rankings[0]["feature"])
print("Revision status: READY_FOR_AI_REVIEW")
print("Approval status:", approval["status"])
print("Prompt saved:", revision_prompt_path)

=== PRODUCTPILOT AI ===
AI REVISION REQUEST PREPARED

Human feedback: Reconsider Calendar because students need better scheduling.
Original recommendation: Deadline Reminders
Revision status: READY_FOR_AI_REVIEW
Approval status: CHANGES_REQUESTED
Prompt saved: productpilot_outputs/revision_request.txt

No Gemini API requests used.


In [ ]:
import json
from pathlib import Path

state_path = Path("productpilot_outputs/workflow_state.json")

with open(state_path, "r", encoding="utf-8") as file:
    state = json.load(file)

# Verify revision request
assert state["approval"]["status"] == "CHANGES_REQUESTED"

assert state["approval"]["approved_feature"] is None

assert state["approval"]["review_feedback"]

# Verify AI review is pending
assert (
    state["workflow"]["revision_status"]
    == "READY_FOR_AI_REVIEW"
)

# Verify revision prompt exists
assert Path(
    "productpilot_outputs/revision_request.txt"
).exists()

# Check that downstream generation is blocked
can_generate_prd = (
    state["approval"]["status"] == "APPROVED"
    and bool(state["approval"]["approved_feature"])
)

assert can_generate_prd is False

print("=== REVISION WORKFLOW VALIDATION ===\n")

print("✅ Human feedback recorded")
print("✅ Previous approval revoked")
print("✅ AI revision request prepared")
print("✅ PRD generation blocked")
print("✅ Revision checkpoint saved")

print("\nRevision workflow preparation PASSED!")

=== REVISION WORKFLOW VALIDATION ===

✅ Human feedback recorded
✅ Previous approval revoked
✅ AI revision request prepared
✅ PRD generation blocked
✅ Revision checkpoint saved

Revision workflow preparation PASSED!


In [ ]:
from agents import Agent

revision_agent = Agent(
    name="Feature Prioritization Agent",

    model=model,

    instructions="""
    You are the Feature Prioritization Agent for ProductPilot AI.

    Your task is to reconsider a product recommendation
    based on feedback from the human product manager.

    Evaluate:
    1. The original feature recommendation.
    2. The human's concerns.
    3. The verified feature rankings.
    4. The trade-offs between alternatives.

    Rules:
    - Do not invent or change priority scores.
    - Distinguish numerical ranking from product judgment.
    - You may recommend a lower-ranked feature if justified.
    - Explain whether you agree with the human's concern.
    - Clearly state your final recommendation.
    - Do not generate a PRD.
    - Do not approve features on the human's behalf.

    StudyFlow and its datasets are fictional.
    """
)

print("Revision agent configured successfully.")

Revision agent configured successfully.


In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone
from agents import Runner

output_dir = Path("productpilot_outputs")
state_path = output_dir / "workflow_state.json"
prompt_path = output_dir / "revision_request.txt"

# Load saved workflow state
with open(state_path, "r", encoding="utf-8") as file:
    state = json.load(file)

# Verify that a human requested changes
if state["approval"]["status"] != "CHANGES_REQUESTED":
    raise PermissionError("No pending revision request.")

if state["workflow"]["revision_status"] != "READY_FOR_AI_REVIEW":
    raise ValueError("Revision is not ready for AI review.")

revision_prompt = prompt_path.read_text(encoding="utf-8")

# Run AI reconsideration
revision_result = await Runner.run(
    revision_agent,
    revision_prompt
)

review_text = revision_result.final_output

if not isinstance(review_text, str) or not review_text.strip():
    raise ValueError("AI returned an empty review.")

# Save the AI's recommendation
review_path = output_dir / "ai_revision_review.md"
review_path.write_text(review_text, encoding="utf-8")

# Update checkpoint
state["workflow"]["revision_status"] = "AWAITING_HUMAN_APPROVAL"
state["workflow"]["revision_review_path"] = str(review_path)
state["updated_at"] = datetime.now(timezone.utc).isoformat()

with open(state_path, "w", encoding="utf-8") as file:
    json.dump(state, file, indent=4)

print("=== AI REVISION REVIEW ===\n")
print(review_text)

print("\n=== WORKFLOW STATUS ===")
print("Revision: AWAITING_HUMAN_APPROVAL")
print("Human approval:", state["approval"]["status"])
print("Review saved:", review_path)

RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 22h58m18.82797052s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See 

In [7]:
from pydantic import BaseModel, Field
from typing import Literal

class FeatureRecommendation(BaseModel):

    recommended_feature: str = Field(
        description="Feature recommended by the AI"
    )

    original_top_feature: str = Field(
        description="Highest-ranked feature from verified calculations"
    )

    decision: Literal[
        "KEEP_ORIGINAL",
        "RECOMMEND_ALTERNATIVE"
    ]

    reasoning: str = Field(
        description="Explanation of the recommendation"
    )

    tradeoffs: list[str] = Field(
        description="Important trade-offs considered"
    )

    requires_human_approval: bool = True


# Test the schema without calling Gemini
sample_recommendation = FeatureRecommendation(
    recommended_feature="Calendar",
    original_top_feature="Deadline Reminders",
    decision="RECOMMEND_ALTERNATIVE",
    reasoning=(
        "Calendar may be preferred when scheduling "
        "is the product manager's primary objective."
    ),
    tradeoffs=[
        "Calendar requires more engineering effort.",
        "Deadline Reminders has the higher calculated priority."
    ]
)

print("=== STRUCTURED OUTPUT TEST ===\n")
print(sample_recommendation.model_dump_json(indent=2))

=== STRUCTURED OUTPUT TEST ===

{
  "recommended_feature": "Calendar",
  "original_top_feature": "Deadline Reminders",
  "decision": "RECOMMEND_ALTERNATIVE",
  "reasoning": "Calendar may be preferred when scheduling is the product manager's primary objective.",
  "tradeoffs": [
    "Calendar requires more engineering effort.",
    "Deadline Reminders has the higher calculated priority."
  ],
  "requires_human_approval": true
}


In [21]:
from agents import Agent
from pydantic import ValidationError

# Reuse our existing FeatureRecommendation schema

structured_revision_agent = Agent(
    name="Feature Prioritization Agent",

    model=model,

    output_type=FeatureRecommendation,

    instructions="""
    You are the Feature Prioritization Agent for ProductPilot AI.

    Reconsider a feature recommendation using:
    - Verified feature rankings
    - Human product manager feedback
    - Engineering trade-offs

    Return a structured recommendation.

    Rules:
    1. Recommend only features present in the supplied rankings.
    2. Never invent or modify numerical priority scores.
    3. Distinguish mathematical ranking from product judgment.
    4. Explain why the recommendation should remain
       unchanged or be revised.
    5. Never approve a feature on behalf of a human.
    6. Always set requires_human_approval to true.

    If you recommend the original highest-ranked feature,
    set decision to KEEP_ORIGINAL.

    Otherwise, set decision to RECOMMEND_ALTERNATIVE.

    StudyFlow and its datasets are fictional.
    """
)

print("Structured revision agent configured.")

Structured revision agent configured.


In [9]:
#cell 53

def validate_feature_recommendation(recommendation, rankings):

    valid_features = {
        item["feature"] for item in rankings
    }

    original_top = rankings[0]["feature"]

    if recommendation.recommended_feature not in valid_features:
        raise ValueError(
            "AI recommended an unknown feature."
        )

    if recommendation.original_top_feature != original_top:
        raise ValueError(
            "Original top feature does not match verified rankings."
        )

    expected_decision = (
        "KEEP_ORIGINAL"
        if recommendation.recommended_feature == original_top
        else "RECOMMEND_ALTERNATIVE"
    )

    if recommendation.decision != expected_decision:
        raise ValueError(
            "Recommendation decision is inconsistent."
        )

    if recommendation.requires_human_approval is not True:
        raise ValueError(
            "Human approval cannot be bypassed."
        )

    return True


# Test with our previously created sample
validate_feature_recommendation(
    sample_recommendation,
    workflow_state["feature_rankings"]
)

print("✅ Structured recommendation validation passed!")

✅ Structured recommendation validation passed!


In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

backup_name = "ProductPilot_AI_Backup.zip"

# Files we want to preserve
project_files = [
    "customer_feedback.csv",
    "competitor_data.csv",
    "engineering_estimates.csv",
    "product_analytics.csv",
]

# Include all generated reports and workflow state
output_dir = Path("productpilot_outputs")

if output_dir.exists():
    project_files.extend(
        str(path)
        for path in output_dir.rglob("*")
        if path.is_file()
    )

# Create ZIP backup
with ZipFile(backup_name, "w", ZIP_DEFLATED) as zip_file:
    for file_path in project_files:
        path = Path(file_path)

        if path.is_file():
            zip_file.write(path, arcname=str(path))
            print("Saved:", path)
        else:
            print("WARNING - Missing:", path)

print("\nBackup created:", backup_name)

# Download to your computer
files.download(backup_name)

Saved: customer_feedback.csv
Saved: competitor_data.csv
Saved: engineering_estimates.csv
Saved: product_analytics.csv
Saved: productpilot_outputs/deadline_reminders_sprint_plan.md
Saved: productpilot_outputs/workflow_state.json
Saved: productpilot_outputs/studyflow_analytics_report.md
Saved: productpilot_outputs/revision_request.txt
Saved: productpilot_outputs/deadline_reminders_prd.md

Backup created: ProductPilot_AI_Backup.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
from google.colab import files
from zipfile import ZipFile

uploaded = files.upload()

backup_name = "ProductPilot_AI_Backup.zip"

with ZipFile(backup_name, "r") as zip_file:
    zip_file.extractall(".")

print("✅ Project files restored successfully!")

Saving ProductPilot_AI_Backup.zip to ProductPilot_AI_Backup.zip
✅ Project files restored successfully!


In [5]:
#Recovery Checkpoint — Run Only After Restoring a Backup

import json
from pathlib import Path

required_files = [
    "customer_feedback.csv",
    "competitor_data.csv",
    "engineering_estimates.csv",
    "product_analytics.csv",
    "productpilot_outputs/workflow_state.json",
    "productpilot_outputs/deadline_reminders_prd.md",
    "productpilot_outputs/deadline_reminders_sprint_plan.md",
    "productpilot_outputs/studyflow_analytics_report.md"
]

missing_files = [
    file for file in required_files
    if not Path(file).exists()
]

if missing_files:
    print("Missing files:", missing_files)
else:
    state_path = Path("productpilot_outputs/workflow_state.json")

    with open(state_path, "r") as f:
        workflow_state = json.load(f)

    print("All project files restored!")
    print("Approval:", workflow_state["approval"]["status"])
    print("Approved feature:", workflow_state["approval"]["approved_feature"])
    print("Revision:", workflow_state["workflow"].get("revision_status"))

All project files restored!
Approval: CHANGES_REQUESTED
Approved feature: None
Revision: READY_FOR_AI_REVIEW


In [10]:
# Test that invalid AI recommendations are rejected

invalid_tests = [
    FeatureRecommendation(
        recommended_feature="Unknown Feature",
        original_top_feature="Deadline Reminders",
        decision="RECOMMEND_ALTERNATIVE",
        reasoning="Invalid feature test",
        tradeoffs=[]
    ),
    FeatureRecommendation(
        recommended_feature="Calendar",
        original_top_feature="Dark Mode",
        decision="RECOMMEND_ALTERNATIVE",
        reasoning="Incorrect original ranking",
        tradeoffs=[]
    ),
    FeatureRecommendation(
        recommended_feature="Calendar",
        original_top_feature="Deadline Reminders",
        decision="KEEP_ORIGINAL",
        reasoning="Incorrect decision test",
        tradeoffs=[]
    )
]

for i, recommendation in enumerate(invalid_tests, start=1):
    try:
        validate_feature_recommendation(
            recommendation,
            workflow_state["feature_rankings"]
        )
        raise AssertionError(f"Test {i} should have failed!")

    except ValueError:
        print(f"PASS: Invalid recommendation {i} rejected.")

print("\nAll negative validation tests passed!")

PASS: Invalid recommendation 1 rejected.
PASS: Invalid recommendation 2 rejected.
PASS: Invalid recommendation 3 rejected.

All negative validation tests passed!


In [11]:
# Verify that human approval cannot be bypassed

unsafe_recommendation = FeatureRecommendation(
    recommended_feature="Calendar",
    original_top_feature="Deadline Reminders",
    decision="RECOMMEND_ALTERNATIVE",
    reasoning="Testing approval protection.",
    tradeoffs=[],
    requires_human_approval=False
)

try:
    validate_feature_recommendation(
        unsafe_recommendation,
        workflow_state["feature_rankings"]
    )

    raise AssertionError("Approval bypass was not blocked!")

except ValueError as error:
    print("PASS: Approval bypass rejected.")
    print("Reason:", error)

print("\nHuman approval protection test passed!")

PASS: Approval bypass rejected.
Reason: Human approval cannot be bypassed.

Human approval protection test passed!


In [12]:
# Final offline validation test
assert validate_feature_recommendation(
    sample_recommendation,
    workflow_state["feature_rankings"]
) is True

assert sample_recommendation.requires_human_approval is True

print("All revision validation checks passed!")

All revision validation checks passed!


In [22]:
def prepare_structured_revision():
    if workflow_state["approval"]["status"] != "CHANGES_REQUESTED":
        raise ValueError("Revision requires a human change request.")

    if workflow_state["workflow"]["revision_status"] != "READY_FOR_AI_REVIEW":
        raise ValueError("Revision request is not ready for AI review.")

    print("Structured revision workflow is ready.")
    print("Agent:", structured_revision_agent.name)
    print("Human approval required:", sample_recommendation.requires_human_approval)

prepare_structured_revision()

Structured revision workflow is ready.
Agent: Feature Prioritization Agent
Human approval required: True


In [15]:
#maybe extra cell

print("Workflow state:")
print(workflow_state)

print("\nSaved state:")
import json
with open("workflow_state.json", "r") as f:
    print(json.load(f))

Workflow state:
{'project': 'ProductPilot AI', 'product': 'StudyFlow', 'data_type': 'synthetic', 'updated_at': '2026-10-26T07:54:45.443728+00:00', 'approval': {'status': 'CHANGES_REQUESTED', 'approved_feature': None, 'review_feedback': 'Reconsider Calendar because students need better scheduling.'}, 'feature_rankings': [{'feature': 'Deadline Reminders', 'score': 8.34}, {'feature': 'Calendar', 'score': 6.75}, {'feature': 'Performance & Stability', 'score': 5.49}, {'feature': 'Dark Mode', 'score': 4.59}, {'feature': 'Cross-Device Sync', 'score': 4.07}, {'feature': 'Study Hour Tracking', 'score': 2.75}, {'feature': 'Study Progress Dashboard', 'score': 2.7}], 'workflow': {'pre_approval_handoffs': 'VERIFIED', 'post_approval_handoffs': 'PENDING_VERIFICATION', 'last_error': 'Gemini daily free-tier quota exceeded', 'revision_status': 'READY_FOR_AI_REVIEW'}, 'reports': {'prd': 'productpilot_outputs/deadline_reminders_prd.md', 'sprint_plan': 'productpilot_outputs/deadline_reminders_sprint_plan.m

FileNotFoundError: [Errno 2] No such file or directory: 'workflow_state.json'

In [23]:
async def run_structured_revision():
    if workflow_state["approval"]["status"] != "CHANGES_REQUESTED":
        raise ValueError("No pending human revision request.")

    if workflow_state["workflow"]["revision_status"] != "READY_FOR_AI_REVIEW":
        raise ValueError("Revision is not ready.")

    with open("productpilot_outputs/revision_request.txt", "r") as f:
        revision_prompt = f.read()

    result = await Runner.run(
        structured_revision_agent,
        revision_prompt
    )

    recommendation = result.final_output

    validate_feature_recommendation(
        recommendation,
        workflow_state["feature_rankings"]
    )

    return recommendation

print("Structured revision function defined.")

Structured revision function defined.


In [24]:
import os

revision_file = "productpilot_outputs/revision_request.txt"

if os.path.exists(revision_file):
    print("Revision request file found.")
    print("Size:", os.path.getsize(revision_file), "bytes")
else:
    print("Revision request file missing.")


Revision request file found.
Size: 1016 bytes


In [25]:
import inspect

assert callable(run_structured_revision)
assert inspect.iscoroutinefunction(run_structured_revision)
assert callable(validate_feature_recommendation)

print("Structured revision function verified.")
print("Async execution:", inspect.iscoroutinefunction(run_structured_revision))
print("Validation enabled:", callable(validate_feature_recommendation))

Structured revision function verified.
Async execution: True
Validation enabled: True


In [26]:
print("Revision status:", workflow_state["workflow"]["revision_status"])
print("Previous API error:", workflow_state["workflow"].get("last_error"))
print("Structured agent ready:", structured_revision_agent.name)

Revision status: READY_FOR_AI_REVIEW
Previous API error: Gemini daily free-tier quota exceeded
Structured agent ready: Feature Prioritization Agent


In [27]:
recommendation = await run_structured_revision()

print("AI revision received!")
print(recommendation.model_dump_json(indent=2))

AI revision received!
{
  "recommended_feature": "Calendar",
  "original_top_feature": "Deadline Reminders",
  "decision": "RECOMMEND_ALTERNATIVE",
  "reasoning": "The human product manager has requested a reconsideration of the 'Calendar' feature due to student needs for better scheduling. While 'Deadline Reminders' holds the highest calculated priority (8.34), the qualitative feedback highlights a critical user pain point that the 'Calendar' feature (ranked 2nd at 6.75) directly addresses. The 'Calendar' feature can offer more comprehensive scheduling and planning capabilities beyond simple deadline alerts. Prioritizing this user need, as expressed by the product manager, justifies recommending 'Calendar' despite its lower calculated rank.",
  "tradeoffs": [
    "Potential decrease in immediate user satisfaction if 'Deadline Reminders' offers a more direct and simpler solution for a subset of users.",
    "Increased development effort and complexity for a robust calendar feature comp

In [28]:
import json
import os

os.makedirs("productpilot_outputs", exist_ok=True)

with open("productpilot_outputs/structured_revision.json", "w") as f:
    json.dump(recommendation.model_dump(), f, indent=4)

workflow_state["workflow"]["revision_status"] = "AWAITING_HUMAN_APPROVAL"

print("Structured recommendation saved.")
print("Revision status:", workflow_state["workflow"]["revision_status"])
print("Recommended feature:", recommendation.recommended_feature)
print("Human approval required:", recommendation.requires_human_approval)


Structured recommendation saved.
Revision status: AWAITING_HUMAN_APPROVAL
Recommended feature: Calendar
Human approval required: True


In [29]:
print("===== HUMAN REVIEW =====")
print("AI recommendation:", recommendation.recommended_feature)
print("Original top-ranked feature:", recommendation.original_top_feature)
print("Decision:", recommendation.decision)
print("\nReasoning:", recommendation.reasoning)

print("\nTrade-offs:")
for item in recommendation.tradeoffs:
    print("-", item)

print("\nHuman approval required:", recommendation.requires_human_approval)

===== HUMAN REVIEW =====
AI recommendation: Calendar
Original top-ranked feature: Deadline Reminders
Decision: RECOMMEND_ALTERNATIVE

Reasoning: The human product manager has requested a reconsideration of the 'Calendar' feature due to student needs for better scheduling. While 'Deadline Reminders' holds the highest calculated priority (8.34), the qualitative feedback highlights a critical user pain point that the 'Calendar' feature (ranked 2nd at 6.75) directly addresses. The 'Calendar' feature can offer more comprehensive scheduling and planning capabilities beyond simple deadline alerts. Prioritizing this user need, as expressed by the product manager, justifies recommending 'Calendar' despite its lower calculated rank.

Trade-offs:
- Potential decrease in immediate user satisfaction if 'Deadline Reminders' offers a more direct and simpler solution for a subset of users.
- Increased development effort and complexity for a robust calendar feature compared to deadline notifications.
-

In [30]:
print("1. Approve Calendar")
print("2. Keep Deadline Reminders")
print("3. Reject both")

choice = input("Enter your decision (1/2/3): ").strip()

if choice == "1":
    workflow_state["approval"]["status"] = "APPROVED"
    workflow_state["approval"]["approved_feature"] = "Calendar"
    workflow_state["workflow"]["revision_status"] = "APPROVED"

elif choice == "2":
    workflow_state["approval"]["status"] = "APPROVED"
    workflow_state["approval"]["approved_feature"] = "Deadline Reminders"
    workflow_state["workflow"]["revision_status"] = "APPROVED"

elif choice == "3":
    workflow_state["approval"]["status"] = "REJECTED"
    workflow_state["approval"]["approved_feature"] = None
    workflow_state["workflow"]["revision_status"] = "REJECTED"

else:
    raise ValueError("Invalid choice. Run this cell again.")

print("\nDecision recorded.")
print("Approval status:", workflow_state["approval"]["status"])
print("Approved feature:", workflow_state["approval"]["approved_feature"])
print("Revision status:", workflow_state["workflow"]["revision_status"])

1. Approve Calendar
2. Keep Deadline Reminders
3. Reject both
Enter your decision (1/2/3): 1

Decision recorded.
Approval status: APPROVED
Approved feature: Calendar
Revision status: APPROVED


In [31]:
import json
import os

os.makedirs("productpilot_outputs", exist_ok=True)

for path in [
    "workflow_state.json",
    "productpilot_outputs/workflow_state.json"
]:
    with open(path, "w") as f:
        json.dump(workflow_state, f, indent=4)

print("Workflow state saved successfully.")
print("Approved feature:", workflow_state["approval"]["approved_feature"])
print("Approval status:", workflow_state["approval"]["status"])

Workflow state saved successfully.
Approved feature: Calendar
Approval status: APPROVED


In [32]:
import json

with open("workflow_state.json", "r") as f:
    saved_state = json.load(f)

assert saved_state["approval"]["status"] == "APPROVED"
assert saved_state["approval"]["approved_feature"] == "Calendar"
assert saved_state["workflow"]["revision_status"] == "APPROVED"

print("Checkpoint verification passed!")
print("Approved feature:", saved_state["approval"]["approved_feature"])
print("Human approval:", saved_state["approval"]["status"])

Checkpoint verification passed!
Approved feature: Calendar
Human approval: APPROVED


In [36]:
prd_path = workflow_state["reports"]["prd"]

with open(prd_path, "r") as f:
    existing_prd = f.read()

print("Approved feature:", workflow_state["approval"]["approved_feature"])
print("Existing PRD contains Calendar:", "Calendar" in existing_prd)
print("Existing PRD contains Deadline Reminders:", "Deadline Reminders" in existing_prd)

Approved feature: Calendar
Existing PRD contains Calendar: False
Existing PRD contains Deadline Reminders: True


In [37]:
import inspect

print("PRD agent available:", "prd_agent" in globals())
print("Save PRD function available:", "save_prd" in globals())

if "save_prd" in globals():
    print("Save PRD signature:", inspect.signature(save_prd))

PRD agent available: True
Save PRD function available: True


TypeError: FunctionTool(name='save_prd', description='Save a Product Requirements Document as a Markdown file.', params_json_schema={'properties': {'feature_name': {'description': 'Name of the approved product feature.', 'title': 'Feature Name', 'type': 'string'}, 'prd_content': {'description': 'Complete PRD text in Markdown format.', 'title': 'Prd Content', 'type': 'string'}}, 'required': ['feature_name', 'prd_content'], 'title': 'save_prd_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7e4061f16e90>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None) is not a callable object

In [38]:
print("Agent name:", prd_agent.name)
print("\nAgent instructions:")
print(prd_agent.instructions)

print("\nAvailable tools:")
for tool in prd_agent.tools:
    print("-", tool.name)

Agent name: PRD Writer Agent

Agent instructions:

    You are the PRD Writer Agent for ProductPilot AI.

    Your responsibility is to write professional Product
    Requirements Documents for approved StudyFlow features.

    StudyFlow is a fictional student productivity application.

    IMPORTANT:
    Only create a PRD when the user explicitly provides
    an approved feature and confirms its approval status.

    Your PRD must contain:

    1. Feature Title
    2. Problem Statement
    3. Product Objective
    4. Target Users
    5. User Stories
    6. Functional Requirements
    7. Nonfunctional Requirements
    8. Acceptance Criteria
    9. Success Metrics
    10. Assumptions and Risks

    Write specific, practical requirements.

    Clearly label proposed targets and assumptions.
    Do not invent verified customer statistics.

    After generating the PRD, you MUST call the
    save_prd tool to save the complete document.

    Return a concise summary and the saved file path.

In [39]:
prd_prompt = f"""
Generate a complete PRD for StudyFlow.

Approved feature: {workflow_state["approval"]["approved_feature"]}
Approval status: {workflow_state["approval"]["status"]}

Human approval has been verified.
Use the PRD structure in your instructions.
Include practical scheduling requirements, user stories,
acceptance criteria, success metrics, assumptions and risks.

Do not use statistics as verified facts unless supplied.
Save the complete PRD using the save_prd tool.
"""

result = await Runner.run(prd_agent, prd_prompt)

print(result.final_output)


The PRD for the StudyFlow Calendar feature has been generated and saved. You can find it at the following path: `productpilot_outputs/calendar_prd.md`.


In [40]:
prd_path = "productpilot_outputs/calendar_prd.md"

with open(prd_path, "r") as f:
    calendar_prd = f.read()

workflow_state["reports"]["prd"] = prd_path

print("PRD exists and is readable:", len(calendar_prd) > 0)
print("PRD length:", len(calendar_prd), "characters")
print("Calendar mentioned:", "Calendar" in calendar_prd)
print("Old PRD preserved:", os.path.exists("productpilot_outputs/deadline_reminders_prd.md"))

PRD exists and is readable: True
PRD length: 6949 characters
Calendar mentioned: True
Old PRD preserved: True


In [41]:
print("Approved feature:", workflow_state["approval"]["approved_feature"])
print("Registered PRD:", workflow_state["reports"]["prd"])
print("PRD file exists:", os.path.exists(workflow_state["reports"]["prd"]))
print("Approval status:", workflow_state["approval"]["status"])

Approved feature: Calendar
Registered PRD: productpilot_outputs/calendar_prd.md
PRD file exists: True
Approval status: APPROVED


In [45]:
print("Sprint agent available:", "sprint_agent" in globals())

Sprint agent available: True


In [46]:
print("Tool:", read_approved_prd.name)
print("Description:", read_approved_prd.description)
print("Approved feature:", workflow_state["approval"]["approved_feature"])
print("PRD path:", workflow_state["reports"]["prd"])
print("Approval status:", workflow_state["approval"]["status"])

Tool: read_approved_prd
Description: Read the saved Product Requirements Document
for an approved StudyFlow feature.
Approved feature: Calendar
PRD path: productpilot_outputs/calendar_prd.md
Approval status: APPROVED


In [58]:
sprint_prompt = f"""
Create a development sprint plan for the approved
StudyFlow feature: {workflow_state["approval"]["approved_feature"]}.

Approval status: {workflow_state["approval"]["status"]}

You MUST call read_approved_prd to retrieve the PRD.
Base the entire sprint plan on that PRD.

Include sprint goals, tasks, descriptions, engineering-day
estimates, dependencies, acceptance criteria, risks,
mitigation and definition of done.

Do not plan unapproved features.
"""

result = await Runner.run(sprint_agent, sprint_prompt)

print(result.final_output)

## 1. Sprint Goal

Implement the core functionality of the StudyFlow Calendar feature, enabling users to view, create, edit, and delete events and tasks, along with basic reminder and categorization capabilities, to improve their schedule management.

## 2. Development Tasks

*   **Task ID: CAL-001**
    *   **Name:** Implement Calendar Data Model
    *   **Description:** Design and implement the database schema for storing events and tasks, including fields for title, date, time, duration, location, description, reminders, and categories.
    *   **Effort Estimate:** 2 engineering days
    *   **Dependencies:** None
    *   **Acceptance Criteria:**
        *   Database schema is created and validated.
        *   Models for events and tasks are defined in the backend.
        *   Basic CRUD (Create, Read, Update, Delete) operations for events and tasks are functional at the data layer.

*   **Task ID: CAL-002**
    *   **Name:** Develop Calendar View Components (Day, Week, Month)
    

In [48]:
approval_status = workflow_state["approval"]["status"]
approved_feature = workflow_state["approval"]["approved_feature"]

print("Approval status:", approval_status)
print("Approved feature:", approved_feature)

Approval status: APPROVED
Approved feature: Calendar


In [51]:
print("Sprint plan generated:", bool(result.final_output))
print("Calendar mentioned:", "Calendar" in result.final_output)
print("Plan length:", len(result.final_output), "characters")
print("\n--- SPRINT PLAN ---\n")
print(result.final_output)

Sprint plan generated: True
Calendar mentioned: True
Plan length: 2029384 characters

--- SPRINT PLAN ---

Sprint Plan: StudyFlow Calendar Feature

**1. Sprint Goal:**
Implement the core functionalities of the StudyFlow Calendar feature, enabling students to view, create, edit, and delete academic and personal events and tasks, along with setting basic reminders and applying color-coding for better organization.

**2. Development Tasks:**

| Task ID | Task Name                     | Description                                                                                                                            | Effort Estimate (Engineering Days) | Dependencies | Acceptance Criteria                                                                                                                                                                                                                                                                                                                

In [52]:
print("Output length:", len(result.final_output))
print("Number of lines:", len(result.final_output.splitlines()))
print("First 500 characters:")
print(result.final_output[:500])

print("\nLast 500 characters:")
print(result.final_output[-500:])

Output length: 2029384
Number of lines: 8
First 500 characters:
Sprint Plan: StudyFlow Calendar Feature

**1. Sprint Goal:**
Implement the core functionalities of the StudyFlow Calendar feature, enabling students to view, create, edit, and delete academic and personal events and tasks, along with setting basic reminders and applying color-coding for better organization.

**2. Development Tasks:**

| Task ID | Task Name                     | Description                                                                                                            

Last 500 characters:
                                                                                                                                                                                                                                                                                                                                                                                                                             

In [53]:
import re

text = result.final_output

print("Total characters:", len(text))
print("Non-whitespace characters:", len(re.sub(r"\s", "", text)))
print("Trailing whitespace:", len(text) - len(text.rstrip()))
print("Longest line:", max(len(line) for line in text.splitlines()))
print("Last 100 characters (repr):", repr(text[-100:]))

Total characters: 2029384
Non-whitespace characters: 381
Trailing whitespace: 2028795
Longest line: 2029047
Last 100 characters (repr): '                                                                                                    '


In [54]:
clean_sprint_plan = result.final_output.rstrip()

print(clean_sprint_plan)
print("\nClean length:", len(clean_sprint_plan))

Sprint Plan: StudyFlow Calendar Feature

**1. Sprint Goal:**
Implement the core functionalities of the StudyFlow Calendar feature, enabling students to view, create, edit, and delete academic and personal events and tasks, along with setting basic reminders and applying color-coding for better organization.

**2. Development Tasks:**

| Task ID | Task Name                     | Description                                                                                                                            | Effort Estimate (Engineering Days) | Dependencies | Acceptance Criteria

Clean length: 589


In [55]:
prd_path = workflow_state["reports"]["prd"]

with open(prd_path, "r", encoding="utf-8") as f:
    prd_text = f.read()

print("PRD characters:", len(prd_text))
print("PRD non-whitespace:", len(prd_text.strip()))
print("PRD trailing spaces:", len(prd_text) - len(prd_text.rstrip()))
print("PRD ends with:", repr(prd_text[-150:]))
print("Sprint agent tools:", [tool.name for tool in sprint_agent.tools])

PRD characters: 6949
PRD non-whitespace: 6948
PRD trailing spaces: 1
PRD ends with: 'ing algorithms) too early could delay launch and impact core functionality. Mitigation: Adhere strictly to the defined scope for the initial release.\n'
Sprint agent tools: ['read_approved_prd']


In [56]:
print(sprint_agent.instructions)


    You are the Sprint Planner Agent for ProductPilot AI.

    Your job is to convert an approved StudyFlow
    Product Requirements Document into a development plan.

    IMPORTANT:
    You MUST call read_approved_prd to retrieve the PRD.
    Never create a sprint plan without reading the PRD.

    Create a development plan containing:

    1. Sprint Goal
    2. Development Tasks
    3. Task Descriptions
    4. Effort Estimates (in engineering days)
    5. Task Dependencies
    6. Acceptance Criteria
    7. Risks and Mitigation
    8. Definition of Done

    REQUIREMENTS:

    - Base tasks on the actual PRD.
    - Make tasks specific and actionable.
    - Include testing and integration tasks.
    - Clearly distinguish estimates from verified facts.
    - Use the provided total engineering estimate as
      a planning constraint.
    - If the estimate seems insufficient, flag the risk.
    - Do not claim that implementation is complete.
    - Do not plan unapproved features.

    Pre

In [59]:
import os

os.makedirs("productpilot_outputs", exist_ok=True)

sprint_plan = result.final_output.strip()

sprint_path = "productpilot_outputs/calendar_sprint_plan.md"

with open(sprint_path, "w", encoding="utf-8") as f:
    f.write(sprint_plan)

print("Sprint plan saved:", sprint_path)
print("Characters:", len(sprint_plan))
print("File exists:", os.path.exists(sprint_path))

Sprint plan saved: productpilot_outputs/calendar_sprint_plan.md
Characters: 7616
File exists: True


In [60]:
import json
import os

# Register the sprint plan
workflow_state["reports"]["sprint_plan"] = sprint_path

# Save both workflow checkpoints
with open("workflow_state.json", "w", encoding="utf-8") as f:
    json.dump(workflow_state, f, indent=4)

with open("productpilot_outputs/workflow_state.json", "w", encoding="utf-8") as f:
    json.dump(workflow_state, f, indent=4)

# Verify
with open("workflow_state.json", "r", encoding="utf-8") as f:
    saved_state = json.load(f)

print("Approved feature:", saved_state["approval"]["approved_feature"])
print("Approval status:", saved_state["approval"]["status"])
print("Sprint plan:", saved_state["reports"]["sprint_plan"])
print("Sprint file exists:", os.path.exists(sprint_path))
print("Checkpoint saved:", os.path.exists("workflow_state.json"))

Approved feature: Calendar
Approval status: APPROVED
Sprint plan: productpilot_outputs/calendar_sprint_plan.md
Sprint file exists: True
Checkpoint saved: True


In [65]:
print("Analytics agent:", analytics_handoff_agent.name)
print("Registered tools:", [tool.name for tool in analytics_handoff_agent.tools])
print("Approved feature:", workflow_state["approval"]["approved_feature"])
print("Approval status:", workflow_state["approval"]["status"])

Analytics agent: Product Analytics Agent
Registered tools: ['calculate_product_metrics']
Approved feature: Calendar
Approval status: APPROVED


In [66]:
analytics_prompt = f"""
The approved StudyFlow feature is:
{workflow_state["approval"]["approved_feature"]}

Approval status:
{workflow_state["approval"]["status"]}

Analyze the existing synthetic product metrics.

You MUST call calculate_product_metrics.
Do not assume the Calendar feature has launched.

Provide verified metrics, trends, three Calendar KPIs,
and recommended next steps.
"""

analytics_result = await Runner.run(
    analytics_handoff_agent,
    analytics_prompt
)

print(analytics_result.final_output)

I have analyzed the synthetic product analytics for StudyFlow. Here's a summary:

**1. Verified Product Metrics (Synthetic Data)**

*   **Weekly Active Users:** Increased from 420 in Week 1 to 510 in Week 4, a growth of 21.43%.
*   **Task Completion Rate:** Saw a gradual increase from 70.00% in Week 1 to 72.44% in Week 4.
*   **Missed Deadline Rate:** Remained relatively stable, fluctuating slightly between 7.39% and 7.60%.
*   **Retention Rate:** Showed a consistent upward trend from 68% in Week 1 to 74% in Week 4.
*   **Total Tasks Created:** 5640
*   **Total Tasks Completed:** 4025
*   **Total Missed Deadlines:** 425
*   **Overall Task Completion Rate:** 71.37%
*   **Overall Missed Deadline Rate:** 7.54%
*   **Average Weekly Retention Rate:** 71.00%
*   **Average Reported Study Hours:** 2.33

**2. Important Trends (Synthetic Data)**

The most notable trend is the consistent growth in active users and retention rates over the four weeks. Alongside this, there's a slight but steady in

In [67]:
import os

analytics_report = analytics_result.final_output.strip()

analytics_path = "productpilot_outputs/calendar_analytics_report.md"

with open(analytics_path, "w", encoding="utf-8") as f:
    f.write(analytics_report)

print("Analytics report saved:", analytics_path)
print("Characters:", len(analytics_report))
print("File exists:", os.path.exists(analytics_path))

Analytics report saved: productpilot_outputs/calendar_analytics_report.md
Characters: 2518
File exists: True


In [68]:
import json
import os

workflow_state["reports"]["analytics"] = analytics_path

with open("workflow_state.json", "w", encoding="utf-8") as f:
    json.dump(workflow_state, f, indent=4)

with open("productpilot_outputs/workflow_state.json", "w", encoding="utf-8") as f:
    json.dump(workflow_state, f, indent=4)

# Verify all deliverables
for name, path in workflow_state["reports"].items():
    if path:
        print(f"{name}: {os.path.exists(path)}")

print("\nApproved feature:", workflow_state["approval"]["approved_feature"])
print("Approval status:", workflow_state["approval"]["status"])
print("Checkpoint saved:", os.path.exists("workflow_state.json"))

prd: True
sprint_plan: True
analytics_report: True
analytics: True

Approved feature: Calendar
Approval status: APPROVED
Checkpoint saved: True


In [69]:
print("=== PRODUCTPILOT FINAL AUDIT ===")

print("\n1. APPROVAL")
print("Feature:", workflow_state["approval"]["approved_feature"])
print("Status:", workflow_state["approval"]["status"])

print("\n2. SAVED REPORTS")
for name, path in workflow_state["reports"].items():
    print(name, "->", path, "| Exists:", os.path.exists(path) if path else False)

print("\n3. AGENT AVAILABILITY")
agents_to_check = [
    "feedback_agent",
    "competitor_agent",
    "prioritization_agent",
    "prd_handoff_agent",
    "sprint_handoff_agent",
    "analytics_handoff_agent"
]

for name in agents_to_check:
    print(name, "->", name in globals())

print("\n4. TOOL AVAILABILITY")
tools_to_check = [
    "read_customer_feedback",
    "get_feedback_statistics",
    "get_competitor_data",
    "calculate_feature_priority",
    "save_prd",
    "read_approved_prd",
    "calculate_product_metrics"
]

for name in tools_to_check:
    print(name, "->", name in globals())

=== PRODUCTPILOT FINAL AUDIT ===

1. APPROVAL
Feature: Calendar
Status: APPROVED

2. SAVED REPORTS
prd -> productpilot_outputs/calendar_prd.md | Exists: True
sprint_plan -> productpilot_outputs/calendar_sprint_plan.md | Exists: True
analytics_report -> productpilot_outputs/studyflow_analytics_report.md | Exists: True
analytics -> productpilot_outputs/calendar_analytics_report.md | Exists: True

3. AGENT AVAILABILITY
feedback_agent -> False
competitor_agent -> False
prioritization_agent -> False
prd_handoff_agent -> True
sprint_handoff_agent -> True
analytics_handoff_agent -> True

4. TOOL AVAILABILITY
read_customer_feedback -> False
get_feedback_statistics -> False
get_competitor_data -> False
calculate_feature_priority -> False
save_prd -> True
read_approved_prd -> True
calculate_product_metrics -> True


In [70]:
print("=== LOADED AGENT/TOOL VARIABLES ===")

for name, value in list(globals().items()):
    if "agent" in name.lower() or (
        "tool" in name.lower() and not name.startswith("_")
    ):
        print(name, "->", type(value).__name__)

=== LOADED AGENT/TOOL VARIABLES ===
Agent -> type
structured_revision_agent -> Agent
function_tool -> function
prd_agent -> Agent
tool -> FunctionTool
sprint_agent -> Agent
analytics_agent -> Agent
analytics_handoff_agent -> Agent
sprint_handoff_agent -> Agent
prd_handoff_agent -> Agent
agents_to_check -> list
tools_to_check -> list


In [71]:
print("=== INITIALIZED AGENTS ===")

for name, value in globals().items():
    if isinstance(value, Agent):
        print(f"\nVariable: {name}")
        print("Agent name:", value.name)
        print("Tools:", [t.name for t in value.tools])
        print("Handoffs:", [a.name for a in value.handoffs])

=== INITIALIZED AGENTS ===

Variable: structured_revision_agent
Agent name: Feature Prioritization Agent
Tools: []
Handoffs: []

Variable: prd_agent
Agent name: PRD Writer Agent
Tools: ['save_prd']
Handoffs: []

Variable: sprint_agent
Agent name: Sprint Planner Agent
Tools: ['read_approved_prd']
Handoffs: []

Variable: analytics_agent
Agent name: Product Analytics Agent
Tools: ['calculate_product_metrics']
Handoffs: []

Variable: analytics_handoff_agent
Agent name: Product Analytics Agent
Tools: ['calculate_product_metrics']
Handoffs: []

Variable: sprint_handoff_agent
Agent name: Sprint Planner Agent
Tools: ['read_approved_prd']
Handoffs: ['Product Analytics Agent']

Variable: prd_handoff_agent
Agent name: PRD Writer Agent
Tools: ['save_prd']
Handoffs: ['Sprint Planner Agent']


In [91]:
# Execute the complete three-agent workflow

workflow_prompt = """
Analyze the StudyFlow customer feedback dataset.

Identify the main customer problems and feedback statistics.
Then research competitor features and gaps.
Finally, calculate feature priorities and recommend
which feature should be developed first.

Use the available tools and pass control through
the configured agent handoffs.

Do not claim that any feature is human-approved.
"""

workflow_result = await Runner.run(
    feedback_handoff_agent,
    workflow_prompt
)

print("=== PRODUCTPILOT AI: COMPLETE AGENT WORKFLOW ===")
print(workflow_result.final_output)

RateLimitError: Error code: 429

In [90]:
required = [
    "model",
    "read_customer_feedback",
    "get_feedback_statistics",
    "get_competitor_data",
    "calculate_feature_priority",
    "prioritization_agent",
    "competitor_handoff_agent",
    "feedback_handoff_agent"
]

for name in required:
    print(f"{name}: {'READY' if name in globals() else 'MISSING'}")

model: READY
read_customer_feedback: READY
get_feedback_statistics: READY
get_competitor_data: READY
calculate_feature_priority: READY
prioritization_agent: READY
competitor_handoff_agent: READY
feedback_handoff_agent: READY


In [92]:
import os

print("Current directory:", os.getcwd())
print("\nProject files:")

for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]
    level = root.count(os.sep)
    if level > 2:
        continue
    print(root, "->", files[:15])

Current directory: /content

Project files:
. -> ['workflow_state.json', 'engineering_estimates.csv', 'product_analytics.csv', 'customer_feedback.csv', 'competitor_data.csv', 'ProductPilot_AI_Backup.zip']
./.config -> ['.last_update_check.json', 'hidden_gcloud_config_universe_descriptor_data_cache_configs.db', '.last_survey_prompt.yaml', 'active_config', 'config_sentinel', 'gce', 'default_configs.db', '.last_opt_in_prompt.yaml']
./.config/configurations -> ['config_default']
./.config/logs -> []
./productpilot_outputs -> ['workflow_state.json', 'deadline_reminders_sprint_plan.md', 'deadline_reminders_prd.md', 'studyflow_analytics_report.md', 'calendar_analytics_report.md', 'revision_request.txt', 'structured_revision.json', 'calendar_sprint_plan.md', 'calendar_prd.md']
./sample_data -> ['README.md', 'anscombe.json', 'california_housing_test.csv', 'mnist_test.csv', 'mnist_train_small.csv', 'california_housing_train.csv']


In [93]:
import zipfile

zip_path = "/content/ProductPilot_AI_Backup.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    for name in z.namelist():
        print(name)

customer_feedback.csv
competitor_data.csv
engineering_estimates.csv
product_analytics.csv
productpilot_outputs/deadline_reminders_sprint_plan.md
productpilot_outputs/workflow_state.json
productpilot_outputs/studyflow_analytics_report.md
productpilot_outputs/revision_request.txt
productpilot_outputs/deadline_reminders_prd.md


In [95]:
import glob

for f in glob.glob("/content/*.ipynb"):
    print(f)